# Where does physics help a graph network? — the whole study, from the original sources

District-level weekly dengue forecasting for Sri Lanka: 25 districts, a 3-week input
window, a 3-week horizon. This notebook is the complete, self-contained reproduction
of the study reported in the paper. It:

1. **verifies** every input file against the SHA-256 recorded when it was retrieved
   from its original source (section 1);
2. **rebuilds** the weekly case series from the Epidemiology Unit's report table, the
   weekly climate from ERA5 daily data, and the population from the Department of
   Census and Statistics PDFs (section 2);
3. **audits** the benchmark array prior work used, to show why it is not used (section 3);
4. **defines** the five published spatio-temporal GNNs and the SEIR-LSTM's LSTM in plain
   PyTorch, the SEIR simulator, and every forecast head (sections 4-5);
5. **trains** every configuration the paper reports under one leakage-controlled
   protocol (sections 6-7); and
6. **analyses** only the rows it has just produced (section 8).

Nothing is loaded from an earlier run: every number printed below is computed here.

## Running it on Kaggle

1. Upload `full_paper/kaggle/dataset/` as a Kaggle dataset (it is built by
   `full_paper/kaggle/fetch_sources.py` and contains only original source files).
2. Create a notebook from this file, attach the dataset, choose a **CPU** accelerator
   (the models are small and the work is many short runs; 4 CPU workers beat a GPU
   here) and "Save Version → Save & Run All".
3. Everything the notebook produces is written to `/kaggle/working/outputs/`.

Internet access is not needed: the one extra package (PyMuPDF, to read the PDFs) is
installed from a wheel inside the dataset.

In [ ]:
# ----------------------------------------------------------------- run settings --
#: "full" runs every configuration the paper reports (several hours on 4 CPUs).
#: "quick" is a smoke test -- one origin, one seed, 3 epochs -- whose numbers are
#: deliberately degraded and must never be quoted.
PROFILE = "full"
#: Parallel worker processes. Kaggle's CPU sessions have 4 cores.
WORKERS = 4

import os
import sys
import time

import matplotlib

if not hasattr(sys, "ps1") and "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch

# Parallel training forks worker processes; forking after PyTorch has started a
# multi-threaded pool can deadlock, and every run is single-threaded anyway.
torch.set_num_threads(1)

NOTEBOOK_START = time.time()
QUICK = PROFILE == "quick"
if QUICK:
    print("PROFILE = 'quick': a smoke test. These numbers are NOT results and must not be quoted.")

## 1. The input files, verified

The dataset holds only files exactly as retrieved: the parsed weekly report table,
one report PDF, the Open-Meteo ERA5 responses, the Census and population PDFs, and
the district graph, boundaries and benchmark array from the benchmark authors'
repository. `SOURCES.csv` lists each file's origin URL and hash; the cell below
recomputes every hash and stops if any file differs.

In [ ]:
import csv
import hashlib
import subprocess
from pathlib import Path


def _find_sources() -> Path:
    """The attached dataset: wherever Kaggle mounts it under /kaggle/input, or a local copy."""
    candidates = [Path("/kaggle/input"), Path.cwd() / "dataset", Path.cwd().parent / "dataset",
                  Path.cwd() / "full_paper" / "kaggle" / "dataset"]
    for base in candidates:
        if base.exists():
            hit = next(iter(sorted(base.rglob("SOURCES.csv"))), None)
            if hit is not None:
                return hit.parent
    raise FileNotFoundError("attach the dataset built by full_paper/kaggle/fetch_sources.py")


SRC = _find_sources()
OUT = Path("/kaggle/working/outputs") if Path("/kaggle/working").exists() else Path.cwd() / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

with (SRC / "SOURCES.csv").open(encoding="utf-8") as f:
    SOURCES = list(csv.DictReader(f))
bad = [r["file"] for r in SOURCES
       if hashlib.sha256((SRC / r["file"]).read_bytes()).hexdigest() != r["sha256"]]
if bad:
    raise SystemExit(f"these files do not match their recorded SHA-256: {bad}")
print(f"{len(SOURCES)} source files verified (SHA-256) in {SRC}")
for r in SOURCES:
    if r["file"].startswith(("population/census2012/", "climate/")):
        continue
    print(f"  {r['file']:58s} {int(r['bytes']):>10,d} B  <- {r['source'][:80]}")
print(f"  + {sum(r['file'].startswith('population/census2012/') for r in SOURCES)} Census 2012 district PDFs"
      f" and {sum(r['file'].startswith('climate/') for r in SOURCES)} ERA5 district responses")
print(f"  files whose hash equals the one recorded at first retrieval: "
      f"{sum(r['manifest_match'] == 'yes' for r in SOURCES)}")

try:
    import pymupdf  # noqa: F401
except ImportError:
    wheel = next((SRC / "wheels").glob("*.whl"))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", str(wheel)], check=True)
    import pymupdf  # noqa: F401
print("PyMuPDF", pymupdf.__version__)

## 2. The data, rebuilt from the sources

### 2.1 Weekly cases

The Epidemiology Unit publishes a Weekly Epidemiological Report (WER); Table 1 gives
dengue cases per Regional Director of Health Services division. The source table is
the parsed Table 1 rows of every report (553 reports, verified against the official
PDFs district by district). The rebuild:

* orders weeks by the report's own **volume and number** (volume N = year 1973 + N),
  not by the parsed date, which is wrong for a handful of reports;
* maps division labels to the 25 districts of the benchmark graph, excluding the
  national total and the Kalmunai division (inside Ampara; the benchmark's config
  excludes it, so the Ampara target is Ampara RDHS only);
* puts every report on a regular grid of report numbers and leaves weeks with **no
  report missing** — never filled, because a filled value is invented data and an
  interpolation borrows the following week;
* applies **one correction**, read from the published PDF itself (below).

In [ ]:
import json
import re

import numpy as np
import pandas as pd

GRAPH = json.loads((SRC / "graph" / "sri_lanka_adj_list.json").read_text(encoding="utf-8"))
NAMES = sorted(GRAPH)                       # district order used everywhere
N = len(NAMES)

#: The dump's truncated or misspelt labels that are not a prefix of their district key.
LABEL_ALIASES = {"paha": "Gampaha", "monaragala": "Moneragala", "nuwara": "NuwaraEliya"}
#: The national total row and the Kalmunai division, excluded as the benchmark excludes them.
EXCLUDED_LABELS = ("srilanka", "94srilanka", "kalmune", "kalmunei", "kalmunai")


def canonical(label: str) -> str | None:
    key = str(label).lower().replace("-", "").replace(" ", "")
    if key in EXCLUDED_LABELS:
        return None
    if key in LABEL_ALIASES:
        return LABEL_ALIASES[key]
    for n in NAMES:
        nk = n.lower().replace("-", "")
        if nk.startswith(key[:6]) or key.startswith(nk[:6]):
            return n
    return None


raw = pd.read_csv(SRC / "cases" / "output_Dengue Fever.csv", encoding="utf-8", encoding_errors="replace")
vol = raw["Source File"].astype(str).str.extract(r"(?i)vol[_ ]?(\d+)[_ ]?no[_ ]?(\d+)")
raw["year"] = vol[0].astype(int) + 1973
raw["week_no"] = vol[1].astype(int)
raw["parsed_date"] = pd.to_datetime(raw["TimeStampStart"], format="%d-%b-%Y %H:%M:%S",
                                    errors="coerce").dt.normalize()
raw["cases"] = pd.to_numeric(raw["Cases"], errors="coerce")
label = raw["Location Name"].astype(str).str.lower().str.replace("-", "").str.replace(" ", "")
national = raw[label.isin(["srilanka", "94srilanka"])].groupby(["year", "week_no"])["cases"].max()
kalmunai = raw[label.isin(["kalmune", "kalmunei", "kalmunai"])].groupby(["year", "week_no"])["cases"].max()
raw["district"] = raw["Location Name"].map(canonical)
dist = raw.dropna(subset=["district"])
assert (dist.groupby(["year", "week_no", "district"])["cases"].nunique() <= 1).all(), \
    "a report-district cell disagrees across source files"
table = dist.groupby(["year", "week_no", "district"]).agg(
    cases=("cases", "first"), parsed_date=("parsed_date", "min")).reset_index()
print(f"source table: {len(raw):,} rows -> {table[['year', 'week_no']].drop_duplicates().shape[0]} reports "
      f"x {table.district.nunique()} districts")

**The week-395 "spike" is a spreadsheet error, corrected from the same report.** In
WER Vol 48 No 02 (26 Dec 2020 – 1 Jan 2021), Table 1's dengue row **A** (cases this
week) has most cells equal to the sum of the two before them — a formula dragged
across the row — and its districts sum to thousands against a printed national
total of 35. Row **B** (cumulative for the year) is internally consistent, and in the
first week of the year cumulative *is* weekly. The cell below reads both rows from
the PDF and replaces row A with row B for that one report.

In [ ]:
import pymupdf

WER_COLUMNS = ["Colombo", "Gampaha", "Kalutara", "Kandy", "Matale", "NuwaraEliya", "Galle",
               "Hambantota", "Matara", "Jaffna", "Kilinochchi", "Mannar", "Vavuniya",
               "Mullaitivu", "Batticaloa", "Ampara", "Trincomalee", "Kurunegala", "Puttalam",
               "Anuradhapura", "Polonnaruwa", "Badulla", "Moneragala", "Ratnapura", "Kegalle",
               "Kalmunai", "SRILANKA"]

page = pymupdf.open(SRC / "cases" / "vol_48_no_02-english_1.pdf")[2]
grid = page.find_tables().tables[0].extract()
header = next(r for r in grid if r and r[0] == "RDHS")
i = next(k for k, r in enumerate(grid) if r and r[0] and str(r[0]).startswith("Dengue"))
assert str(grid[i][1]) == "B" and str(grid[i + 1][1]) == "A", "unexpected row layout in the Dengue block"
assert len(header[2:]) == len(WER_COLUMNS)
row_b = [int(x) for x in grid[i][2:]]
row_a = [int(x) for x in grid[i + 1][2:]]
formula_cells = sum(1 for k in range(2, 26) if abs(row_a[k] - (row_a[k - 1] + row_a[k - 2])) <= 1)
assert sum(row_b[:25]) + row_b[25] == row_b[26], "row B does not add up; refusing to use it"
print(f"row A (printed 'cases this week'): districts sum {sum(row_a[:25]):,}, national cell {row_a[26]}, "
      f"{formula_cells}/24 cells = sum of the two before")
print(f"row B (year to date = week 1):     districts + Kalmunai = {sum(row_b[:25]) + row_b[25]} "
      f"= national {row_b[26]}")
CORRECTION = {(2021, 2): dict(zip(WER_COLUMNS[:25], row_b[:25]))}

In [ ]:
def report_dates(tab: pd.DataFrame) -> pd.Series:
    """One start date per report; a parsed date whose year disagrees with the volume is re-derived."""
    per = tab.groupby(["year", "week_no"])["parsed_date"].min()
    ok = (per.dt.year == per.index.get_level_values("year")) | (
        (per.dt.month == 12) & (per.index.get_level_values("year") == per.dt.year + 1))
    good, keys, fixed = per[ok], list(per.index), per.copy()
    for i, k in enumerate(keys):
        if ok.loc[k]:
            continue
        for step in range(1, len(keys)):
            hit = next((j for j in (i - step, i + step) if 0 <= j < len(keys) and keys[j] in good.index), None)
            if hit is not None:
                fixed.loc[k] = good.loc[keys[hit]] + pd.Timedelta(weeks=i - hit)
                break
    return fixed


tab = table.copy()
for (y, n), repl in CORRECTION.items():
    mask = (tab.year == y) & (tab.week_no == n)
    for d, v in repl.items():
        tab.loc[mask & (tab.district == d), "cases"] = v
last_no = tab.groupby("year")["week_no"].max()
first_year, last_year = int(tab.year.min()), int(tab.year.max())
first_no = int(tab.loc[tab.year == first_year, "week_no"].min())
week_grid = []
for y in range(first_year, last_year + 1):
    n_weeks = max(52, int(last_no.get(y, 52)))
    start = first_no if y == first_year else 1
    stop = int(last_no[y]) if y == last_year else n_weeks
    week_grid.extend((y, n) for n in range(start, stop + 1))
wide = (tab.pivot_table(index=["year", "week_no"], columns="district", values="cases", aggfunc="first")
        .reindex(pd.MultiIndex.from_tuples(week_grid, names=["year", "week_no"])).reindex(columns=NAMES))
observed = wide.notna().all(axis=1).to_numpy()
dates = report_dates(tab).reindex(wide.index)
if dates.isna().any():
    dates = dates.interpolate()          # dates only: the case values of a missing week stay missing
if dates.isna().any():                   # a missing week at either end: step from the nearest known date
    known = dates.dropna()
    pos = {k: i for i, k in enumerate(wide.index)}
    for k in dates[dates.isna()].index:
        j = min(known.index, key=lambda kk: abs(pos[kk] - pos[k]))
        dates.loc[k] = known.loc[j] + pd.Timedelta(weeks=pos[k] - pos[j])
CASES = wide.to_numpy(dtype=np.float64)
MISSING = ~observed
WEEK_START = pd.Series(pd.to_datetime(dates.to_numpy()))
YEARS = np.array([k[0] for k in wide.index])
T = len(CASES)

check = wide.sum(axis=1).to_frame("districts").join(
    pd.DataFrame({"national": national, "kalmunai": kalmunai}))
check = check[observed & check.national.notna().to_numpy()]
gap = (check.districts + check.kalmunai.fillna(0) - check.national).abs()
print(f"rebuilt series: {T} weeks, {week_grid[0][0]}-W{week_grid[0][1]} .. {week_grid[-1][0]}-W{week_grid[-1][1]}, "
      f"{MISSING.sum()} weeks with no report (left missing)")
print(f"integrity: districts + Kalmunai equal the Unit's own national total in "
      f"{int((gap == 0).sum())} of {len(check)} reports")

### 2.2 Weekly climate (ERA5)

Six daily ERA5 variables per district, from the Open-Meteo archive at one interior
point per district (the centroid if it lies inside the district's GADM polygon,
otherwise a representative point). Each report week takes the mean of its 7 days,
precipitation the 7-day total. **No lag is baked in**: the model reads climate only
up to two weeks before the forecast week (ERA5 is published about 5 days late), and
that rule is applied when windows are cut, not here.

In [ ]:
CLIMATE_VARS = ["temperature_2m_mean", "temperature_2m_min", "temperature_2m_max",
                "precipitation_sum", "relative_humidity_2m_mean", "soil_moisture_0_to_7cm_mean"]
CLIMATE = np.full((T, N, len(CLIMATE_VARS)), np.nan)
for j, name in enumerate(NAMES):
    daily = pd.DataFrame(json.loads((SRC / "climate" / f"{name}.json").read_text(encoding="utf-8"))["daily"])
    daily["time"] = pd.to_datetime(daily["time"])
    daily = daily.set_index("time")
    for t, ws in enumerate(WEEK_START):
        span = daily.loc[ws: ws + pd.Timedelta(days=6)]
        for c, v in enumerate(CLIMATE_VARS):
            CLIMATE[t, j, c] = span[v].sum() if v == "precipitation_sum" else span[v].mean()
assert not np.isnan(CLIMATE).any(), "ERA5 does not cover every week and district"
print(f"climate: {CLIMATE.shape} (weeks, districts, variables) from {N} daily ERA5 series")

### 2.3 Population

The SEIR model needs each district's population. A mid-year estimate for year Y is
published after year Y, so each week in year Y uses the **latest official figure for
an earlier year**: the 2012 Census count for 2013-2014, and the DCS mid-year estimate
for Y − 1 from 2015 on. Both are read from the official PDFs. Each Census total must
equal male + female on the same line and the 25 must sum to the national census total.

In [ ]:
DCS_TO_GRAPH = {"Nuwara-eliya": "NuwaraEliya", "Monaragala": "Moneragala"}
rows = []
for pg in pymupdf.open(SRC / "population" / "Mid-year_population_by_district_and_sex_2024.pdf"):
    for tbl in pg.find_tables().tables:
        g = tbl.extract()
        years = [(k, str(c)) for k, c in enumerate(g[0]) if c and str(c)[:4].isdigit()]
        for line in g[2:]:
            nm = (line[0] or "").strip()
            for col, lab in years:
                if nm and line[col] not in (None, ""):
                    rows.append({"district": DCS_TO_GRAPH.get(nm, nm), "year": int(lab[:4]),
                                 "thousands": int(str(line[col]).replace(",", ""))})
midyear = pd.DataFrame(rows).drop_duplicates(["district", "year"])
for y, g in midyear.groupby("year"):
    nat = int(g.loc[g.district == "Sri Lanka", "thousands"].iloc[0])
    assert abs(nat - int(g.loc[g.district != "Sri Lanka", "thousands"].sum())) <= 13, y
midyear = midyear[midyear.district != "Sri Lanka"].pivot(index="year", columns="district",
                                                         values="thousands")[NAMES] * 1000

DCS_SPELLING = {"Batticaloa": "Baticaloa", "Moneragala": "Monaragala"}
SEPARATE_A1 = {"Anuradhapura", "Moneragala", "Polonnaruwa"}
census = {}
for name in NAMES:
    sp = DCS_SPELLING.get(name, name)
    pdf = SRC / "population" / "census2012" / (f"{sp}_A1.pdf" if name in SEPARATE_A1 else f"{sp}.pdf")
    found = None
    for pg in pymupdf.open(pdf):
        text = pg.get_text().replace("\xa0", " ")
        for lab in ("District", re.escape(name), re.escape(sp)):
            for m in re.finditer(lab, text):
                nums = [int(x.replace(",", "")) for x in re.findall(r"\d[\d,]*", text[m.end(): m.end() + 400])]
                if len(nums) >= 3 and nums[0] > 10_000 and nums[0] == nums[1] + nums[2]:
                    found = nums[0]
                    break
            if found:
                break
        if found:
            break
    assert found, f"{name}: no total = male + female line in {pdf.name}"
    census[name] = found
NATIONAL_CENSUS_2012 = 20_359_439   # DCS, Census of Population and Housing 2012 - Key Findings, p. 19
assert sum(census.values()) == NATIONAL_CENSUS_2012, "district census totals do not sum to the national total"

pop_rows = []
for y in YEARS:
    earlier = [yy for yy in midyear.index if yy < y]
    pop_rows.append(midyear.loc[max(earlier)].to_numpy(float) if earlier
                    else np.array([census[n] for n in NAMES], float))
POPULATION = np.stack(pop_rows)
print(f"population: census 2012 sums to {sum(census.values()):,}; mid-year estimates "
      f"{midyear.index.min()}-{midyear.index.max()}; each week uses an earlier year's figure")

### 2.4 The no-future-information rule, and the evaluation protocol

A forecast is made at the start of week *i* for weeks *i..i+2*. It may read cases up
to week *i−1* and climate up to week *i−2*; population is already an earlier year's
figure. **Folds** follow the frozen protocol: rolling origins at 0.55, 0.70 and 0.85 of
the window list (and nine disjoint origins 0.40 + k/15 for confirmation), the 30
windows before each origin for validation, the next 15% (or 1/15) for test, and
normalisation statistics from training weeks only. Windows touching a missing week
are dropped; fold boundaries are computed before dropping, so a gap never shifts them.

In [ ]:
import torch

LAGS = {"cases": 1, "climate": 2}
WINDOW, HORIZON = 3, 3
ORIGINS = (0.55, 0.70, 0.85)
TEST_FRAC = 0.15
ORIGINS_9 = tuple(round(0.40 + k / 15.0, 4) for k in range(9))
TEST_FRAC_9 = 1.0 / 15.0

from dataclasses import dataclass, replace as dc_replace


@dataclass
class Fold:
    origin: float
    idx: dict
    mean: float
    std: float
    window: int = WINDOW


def build_folds(origins=ORIGINS, test_frac=TEST_FRAC, window=WINDOW) -> list[Fold]:
    bad = set(np.where(MISSING)[0].tolist())
    ids = list(range(window, T - HORIZON))
    clean = lambda i: not any(t in bad for t in range(i - window, i + HORIZON))  # noqa: E731
    folds = []
    for o in origins:
        cut, end = int(o * len(ids)), int(min(o + test_frac, 1.0) * len(ids))
        tr = [i for i in ids[: cut - 30] if clean(i)]
        va = [i for i in ids[cut - 30: cut] if clean(i)]
        te = [i for i in ids[cut:end] if clean(i)]
        hist = np.log1p(CASES[: ids[: cut - 30][-1] + 1])
        folds.append(Fold(o, {"train": np.array(tr), "val": np.array(va), "test": np.array(te)},
                          float(np.nanmean(hist)), float(np.nanstd(hist) + 1e-8), window))
    return folds


def seasonal_features(idx: np.ndarray) -> np.ndarray:
    """sin/cos of day-of-year at one and two cycles per year, (K, 4)."""
    ang = 2 * np.pi * WEEK_START.dt.dayofyear.to_numpy()[idx] / 365.25
    return np.stack([np.sin(ang), np.cos(ang), np.sin(2 * ang), np.cos(2 * ang)], -1)


def with_history(fold: Fold, weeks: int) -> Fold:
    return dc_replace(fold, idx={k: v[v - weeks >= 0] for k, v in fold.idx.items()})


def climate_blocks(idx: np.ndarray, blocks, base: np.ndarray) -> np.ndarray:
    """Means of the climate over lag blocks (inclusive week ranges before the origin)."""
    if len(idx) and int(np.min(idx)) - max(b for _, b in blocks) < 0:
        raise ValueError("a window has no climate that far back; drop it with with_history()")
    out = []
    for a, b in blocks:
        assert a >= LAGS["climate"], "climate closer than the ERA5 release delay is not available"
        out.append(np.stack([base[idx - lag] for lag in range(a, b + 1)], 0).mean(0))
    return np.concatenate(out, -1)


def build_tensors(fold: Fold, split: str, use_climate=False, use_season=False, clim_blocks=()) -> dict:
    idx = fold.idx[split]
    x_raw = np.stack([CASES[i - fold.window: i].T for i in idx])
    y_raw = np.stack([CASES[i: i + HORIZON].T for i in idx])
    p_raw = np.repeat(CASES[idx - 1][:, :, None], HORIZON, axis=2)
    z = lambda a: (np.log1p(a) - fold.mean) / fold.std  # noqa: E731
    feats = [z(x_raw)]
    if use_climate:
        lo = LAGS["climate"]
        cut = lambda ids: np.stack([np.moveaxis(CLIMATE[i - lo - 2: i - lo + 1], 0, 1) for i in ids])  # noqa: E731
        cl, ref = cut(idx), cut(fold.idx["train"])
        m, s = ref.mean((0, 1, 2)), ref.std((0, 1, 2)) + 1e-8
        feats.append(((cl - m) / s).reshape(len(idx), N, -1))
    if clim_blocks:
        blk, ref = climate_blocks(idx, clim_blocks, CLIMATE), climate_blocks(fold.idx["train"], clim_blocks, CLIMATE)
        feats.append((blk - ref.mean((0, 1))) / (ref.std((0, 1)) + 1e-8))
    if use_season:
        feats.append(np.repeat(seasonal_features(idx)[:, None, :], N, axis=1))
    t = lambda a: torch.tensor(np.asarray(a), dtype=torch.float32)  # noqa: E731
    return {"x": t(np.concatenate(feats, -1)), "y_raw": t(y_raw), "p_raw": t(p_raw), "y_z": t(z(y_raw)),
            "p_z": t(z(p_raw)), "x_raw": t(x_raw), "pop": t(POPULATION[idx - 1]), "idx": idx}


def rmse(pred, truth) -> float:
    return float(np.sqrt(np.mean((np.asarray(pred) - np.asarray(truth)) ** 2)))


def score(pred, truth) -> dict:
    out = {"RMSE": rmse(pred, truth), "MAE": float(np.mean(np.abs(pred - truth)))}
    for h in range(truth.shape[-1]):
        out[f"RMSE_h{h + 1}"] = rmse(pred[..., h], truth[..., h])
    return out


# The district graph: the benchmark's adjacency list with self-loops. Edges are listed
# row-major (this order matters to DCRNN, section 4).
_a = np.eye(N, dtype=np.float32)
for d, nbrs in GRAPH.items():
    for o in nbrs:
        if o in NAMES:
            _a[NAMES.index(d), NAMES.index(o)] = 1.0
_src, _dst = np.nonzero(_a)
EDGE_INDEX = torch.tensor(np.stack([_src, _dst]), dtype=torch.long)
FIXED = torch.tensor(_a / _a.sum(1, keepdims=True), dtype=torch.float32)   # row-normalised
print(f"graph: {N} districts, {int(_a.sum()) - N} directed edges plus self-loops, "
      f"{int((_a != _a.T).sum())} one-way entries")

for f in build_folds():
    print(f"origin {f.origin}: train {len(f.idx['train'])}, val {len(f.idx['val'])}, test {len(f.idx['test'])} "
          f"windows; test weeks {WEEK_START[f.idx['test'][0]]:%Y-%m-%d} .. {WEEK_START[f.idx['test'][-1]]:%Y-%m-%d}")

## 3. Why not the benchmark array?

Prior work, including the benchmark and our own first phase, used the processed array
`sri_lanka_2013-2022_shifted.npy` from the benchmark authors' repository: 459 rows
(weeks) × 25 districts × 11 channels (5 weather channels at lag 0, the cases, and 5
channels documented as lagged 12 or 17 weeks). The array has no date axis. This
section dates it and measures what that shows.

In [ ]:
ARR = np.load(SRC / "benchmark" / "sri_lanka_2013-2022_shifted.npy", allow_pickle=True).astype(float)
arr_cases = np.nan_to_num(ARR[..., 5])
print(f"benchmark array: {ARR.shape} (rows, districts, channels)")

# Date every array row by matching its 25 district counts exactly against the report
# weeks. The array was built from the published row A, so the comparison uses the
# report values before our one correction.
published = (table.pivot_table(index=["year", "week_no"], columns="district", values="cases", aggfunc="first")
             .reindex(pd.MultiIndex.from_tuples(week_grid, names=["year", "week_no"])).reindex(columns=NAMES)
             .to_numpy(float))
lookup: dict[bytes, list[int]] = {}
for t in range(T):
    if not np.isnan(published[t]).any():
        lookup.setdefault(published[t].tobytes(), []).append(t)
row_week = np.array([lookup.get(arr_cases[k].tobytes(), [-1])[0] for k in range(len(ARR))])
matched = row_week >= 0
row_date = pd.Series([WEEK_START[t] if t >= 0 else pd.NaT for t in row_week])
row_year = row_date.dt.year

dups = [k for k in range(len(ARR) - 1) if arr_cases[k].sum() > 0 and np.array_equal(arr_cases[k], arr_cases[k + 1])]
print(f"rows matched exactly to a published report: {matched.sum()} of {len(ARR)}; unmatched rows: "
      f"{np.flatnonzero(~matched).tolist()}")
print(f"identical consecutive rows (one week stored twice): {dups}")
print(f"rows 0-47 are dated {row_date[:48].min():%Y-%m-%d} .. {row_date[:48].max():%Y-%m-%d}; "
      f"rows 51 onward run {row_date[51:].min():%Y-%m-%d} .. {row_date[51:].max():%Y-%m-%d}")
span = (WEEK_START >= row_date[51:].min()) & (WEEK_START <= row_date[51:].max()) & pd.Series(~MISSING)
used = set(row_week[matched].tolist())
print(f"reports inside the array's 2013-2022 span that never reached it: "
      f"{sum(1 for t in np.flatnonzero(span.to_numpy()) if t not in used)}")

# The benchmark protocol's folds on the array (rolling origins over its row order).
ids = list(range(3, len(ARR) - 3))
for o in ORIGINS:
    cut, end = int(o * len(ids)), int(min(o + 0.15, 1.0) * len(ids))
    tr, te = ids[: cut - 30], ids[cut:end]
    print(f"  benchmark fold {o}: {sum(1 for r in tr if row_year[r] == 2023)} rows from 2023 in TRAINING; "
          f"test rows dated {row_date[te].min():%Y-%m} .. {row_date[te].max():%Y-%m}")

**The climate columns are on a different timeline, and the "lagged" ones point into
the future.** The weather channels were not reordered with the cases, so they are
dated separately: the cell searches for the calendar offset at which the array's
temperature best matches ERA5 temperature. It then tests the precipitation channel
documented as "lag 12" — which should hold rain from 12 weeks *earlier* — at every
shift from −20 to +20 weeks.

In [ ]:
era_daily = {}
for name in NAMES:
    d = pd.DataFrame(json.loads((SRC / "climate" / f"{name}.json").read_text(encoding="utf-8"))["daily"])
    era_daily[name] = d.assign(time=pd.to_datetime(d["time"])).set_index("time")
no_jaffna = [j for j, n in enumerate(NAMES) if n != "Jaffna"]     # the array's Jaffna weather is all zero


def era_weekly(var: str, starts: pd.DatetimeIndex, total: bool = False) -> np.ndarray:
    out = np.full((len(starts), N), np.nan)
    for j, name in enumerate(NAMES):
        s = era_daily[name][var]
        roll = s.rolling(7).sum() if total else s.rolling(7).mean()     # value at day t covers t-6..t
        out[:, j] = roll.reindex(starts + pd.Timedelta(days=6)).to_numpy()
    return out


def corr(a: np.ndarray, b: np.ndarray) -> float:
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.corrcoef(a[m], b[m])[0, 1])


arr_temp = ARR[:, no_jaffna, 0].mean(1)
best = None
for off in range(-120, 121):          # candidate first-row dates around the dating implied by the reports
    d0 = pd.Timestamp("2013-05-17") + pd.Timedelta(days=off)
    starts = pd.DatetimeIndex([d0 + pd.Timedelta(weeks=k) for k in range(len(ARR))])
    r = corr(arr_temp, era_weekly("temperature_2m_mean", starts)[:, no_jaffna].mean(1))
    if best is None or r > best[0]:
        best = (r, d0)
r_best, D0 = best
clim_starts = pd.DatetimeIndex([D0 + pd.Timedelta(weeks=k) for k in range(len(ARR))])
case_starts = pd.DatetimeIndex([row_date[k] if matched[k] else pd.NaT for k in range(len(ARR))])
r_case = corr(arr_temp[matched], era_weekly("temperature_2m_mean",
                                           case_starts[matched])[:, no_jaffna].mean(1))
print(f"array temperature vs ERA5: best at climate row 0 = {D0:%Y-%m-%d} (r = {r_best:.3f}); "
      f"on the dates the CASE rows carry, r = {r_case:.3f}")

prec = ARR[:, no_jaffna, 7].ravel()
shifts = list(range(-20, 21))
r_shift = [corr(prec, era_weekly("precipitation_sum", clim_starts + pd.Timedelta(weeks=s), total=True)
                [:, no_jaffna].ravel()) for s in shifts]
peak = shifts[int(np.nanargmax(r_shift))]
print(f"'lag 12' precipitation channel vs ERA5 rain {'later' if peak > 0 else 'earlier'} by k weeks: "
      f"best at k = {peak:+d} (r = {max(r_shift):.3f}); at the documented k = -12, r = {r_shift[shifts.index(-12)]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].plot(case_starts, arr_temp, ".", ms=3, label="array temperature, at the case rows' dates")
axes[0].plot(clim_starts, arr_temp, "-", lw=0.8, label="array temperature, re-dated")
axes[0].legend(fontsize=7); axes[0].set_title("the climate columns were not reordered with the cases")
axes[1].bar(shifts, r_shift, color=["#e76f51" if s > 0 else "#9aa5b1" for s in shifts])
axes[1].axvline(-12, color="k", ls="--", lw=1)
axes[1].set_xlabel("ERA5 rainfall shifted by k weeks (negative = earlier, as documented)")
axes[1].set_ylabel("correlation"); axes[1].set_title("the 'lag 12' rain channel holds FUTURE rain")
fig.tight_layout(); fig.savefig(OUT / "audit.png", dpi=150); plt.show()

**The largest spike is the spreadsheet error.** Row 395 of the array carries the
printed row A of WER Vol 48 No 02, not the true count.

In [ ]:
k395 = int(np.flatnonzero(row_week == int(np.flatnonzero(
    (np.array([k[0] for k in wide.index]) == 2021) & (np.array([k[1] for k in wide.index]) == 2))[0]))[0])
print(f"array row {k395}: districts sum {arr_cases[k395].sum():,.0f} "
      f"(printed row A {sum(row_a[:25]):,}); the corrected week sums to {sum(row_b[:25]):,}")

# The persistence floor on the array, under the benchmark protocol, with and without
# the windows that touch the spreadsheet error.
floors = []
for o in ORIGINS:
    cut, end = int(o * len(ids)), int(min(o + 0.15, 1.0) * len(ids))
    te = ids[cut:end]
    err = np.stack([np.repeat(arr_cases[i - 1][:, None], 3, 1) - arr_cases[i:i + 3].T for i in te])
    touch = np.array([k395 in range(i - 3, i + 3) for i in te])
    floors.append((rmse(err, 0), rmse(err[~touch], 0)))
ARRAY_FLOOR = np.mean(floors, axis=0)
print(f"persistence on the array (mean over the 3 origins): {ARRAY_FLOOR[0]:.2f} over all test windows, "
      f"{ARRAY_FLOOR[1]:.2f} without the windows touching row {k395}")

## 4. The six encoders, written out

The five published spatio-temporal GNNs (Weng et al. 2024) and the LSTM of the
SEIR-LSTM (Liu et al. 2025), in plain PyTorch. Nothing is imported from a graph
library: with 25 districts every graph operation is a small dense matrix
product, so each layer is written as the equation it implements.

Each one is a **transcription** of the implementation the benchmark used
(`torch_geometric_temporal` 0.54 and PyTorch Geometric's `GATConv`), wired the
way the benchmark wires it. Parameter names match those implementations, so a
weight-for-weight comparison is possible; the repository test
`tests/test_kaggle_architectures.py` loads the reference weights into every
module below and checks the outputs agree to floating-point precision.

Two properties of the reference implementations are kept on purpose, because
they are part of the published models' behaviour:

* **A3TGCN** restarts its GRU state at zero for every input week and
  attention-weights the per-week outputs; it is not a recurrence over the window.
* **DCRNN**'s diffusion convolution pairs its reverse-direction weights with the
  edge list *by position*, and its Chebyshev-style recursion resets the older
  term to the input at every step. Both are reproduced exactly.

Each encoder maps a window of fold-scaled log cases `(batch, 25, 3)` to a
representation `(batch, 25, 64)` that the forecast heads (section 5) read.

In [ ]:
import math

import torch
import torch.nn.functional as F
from torch import nn


def glorot_(t: torch.Tensor) -> torch.Tensor:
    """PyG's glorot: uniform in +/- sqrt(6 / (fan_in + fan_out)) on the last two dims."""
    stdv = math.sqrt(6.0 / (t.size(-2) + t.size(-1)))
    with torch.no_grad():
        return t.uniform_(-stdv, stdv)


def dense_adjacency(edge_index: torch.Tensor, n: int) -> torch.Tensor:
    """``A[src, dst] = 1`` for every edge; duplicates are not expected."""
    a = torch.zeros(n, n)
    a[edge_index[0], edge_index[1]] = 1.0
    return a


class Head(nn.Module):
    """The reference's single output layer (``improved.make_head`` with no increments).

    ``nodes == 0``: applied to every node, ``(..., in) -> (..., out)``.
    ``nodes > 0`` : emits every node from one graph-level vector (STGAT), ``(B, in) -> (B, nodes, out)``.
    """

    def __init__(self, in_features: int, out: int, nodes: int = 0):
        super().__init__()
        self.out, self.nodes = out, nodes
        self.linear = nn.Linear(in_features, max(nodes, 1) * out)

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        y = self.linear(h)
        return y.reshape(*h.shape[:-1], self.nodes, self.out) if self.nodes else y

### STGAT — graph attention over districts, then two LSTMs over time

A graph-attention layer (8 heads, averaged) mixes each week's case values
across neighbouring districts:

$\alpha_{ij} = \mathrm{softmax}_{j \in \mathcal{N}(i) \cup \{i\}}\big(\mathrm{LeakyReLU}(a_s^\top W x_j + a_d^\top W x_i)\big)$,
$\;x_i' = \tfrac{1}{8}\sum_{\text{heads}} \sum_j \alpha_{ij} W x_j + b$.

The attended window then enters two LSTMs **with the 25 districts as the feature
axis**, so the LSTM output is one graph-level vector per sample, and a linear layer
emits every district from it — the benchmark's wiring.

In [ ]:
class GATConv(nn.Module):
    """PyG ``GATConv(in, out, heads, concat=False, dropout)`` on a dense 25-node graph."""

    def __init__(self, in_channels: int, out_channels: int, heads: int = 8, dropout: float = 0.0,
                 negative_slope: float = 0.2):
        super().__init__()
        self.heads, self.out_channels = heads, out_channels
        self.dropout, self.negative_slope = dropout, negative_slope
        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.att_src = nn.Parameter(torch.empty(1, heads, out_channels))
        self.att_dst = nn.Parameter(torch.empty(1, heads, out_channels))
        self.bias = nn.Parameter(torch.empty(out_channels))
        glorot_(self.lin.weight)
        glorot_(self.att_src)
        glorot_(self.att_dst)
        nn.init.zeros_(self.bias)

    def forward(self, x: torch.Tensor, incoming: torch.Tensor) -> torch.Tensor:
        """``x`` (B, N, in); ``incoming[i, j] = 1`` if j sends to i (self-loops included)."""
        b, n, _ = x.shape
        xp = self.lin(x).view(b, n, self.heads, self.out_channels)            # (B,N,H,C)
        a_src = (xp * self.att_src).sum(-1)                                   # (B,N,H)
        a_dst = (xp * self.att_dst).sum(-1)
        e = F.leaky_relu(a_dst[:, :, None, :] + a_src[:, None, :, :], self.negative_slope)  # (B,i,j,H)
        e = e.masked_fill(incoming[None, :, :, None] == 0, float("-inf"))
        alpha = F.dropout(torch.softmax(e, dim=2), p=self.dropout, training=self.training)
        out = torch.einsum("bijh,bjhc->bihc", alpha, xp)
        return out.mean(dim=2) + self.bias


class STGAT(nn.Module):
    def __init__(self, n_nodes: int, window: int, out: int, edge_index: torch.Tensor,
                 heads: int = 8, hidden: int = 64, dropout: float = 0.1):
        super().__init__()
        self.dropout = dropout
        self.gat = GATConv(window, window, heads=heads, dropout=dropout)
        self.lstm1 = nn.LSTM(n_nodes, hidden, num_layers=1)
        self.lstm2 = nn.LSTM(hidden, hidden, num_layers=1)
        self.head = Head(hidden, out, nodes=n_nodes)
        for lstm in (self.lstm1, self.lstm2):
            for name, p in lstm.named_parameters():
                (nn.init.constant_(p, 0.0) if "bias" in name else nn.init.xavier_uniform_(p))
        nn.init.xavier_uniform_(self.head.linear.weight)
        a = dense_adjacency(edge_index, n_nodes)
        incoming = a.T.clone()                     # incoming[i, j] = A[j, i]
        incoming.fill_diagonal_(1.0)               # GATConv removes then re-adds self-loops
        self.register_buffer("incoming", incoming)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n, w = x.shape
        h = F.dropout(self.gat(x, self.incoming), self.dropout, training=self.training)
        h = h.movedim(2, 0)                        # (window, B, N): districts are the LSTM features
        h, _ = self.lstm1(h)
        h, _ = self.lstm2(h)
        return self.head(h[-1])

### A3TGCN — a graph-convolutional GRU per week, attention over weeks

The GCN is $\tilde{A} X W + b$ with $\tilde{A} = D^{-1/2} A^\top D^{-1/2}$ over the
graph with self-loops and $D$ the in-degree. The TGCN cell is a GRU whose gates
are GCNs of the input concatenated with the state. A3TGCN runs that cell on each
of the 3 input weeks from a zero state and combines the results with
softmax-normalised learned weights over weeks.

In [ ]:
class GCNConv(nn.Module):
    """PyG ``GCNConv`` (symmetric normalisation, self-loops) on a dense graph."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.lin = nn.Linear(in_channels, out_channels, bias=False)
        self.bias = nn.Parameter(torch.zeros(out_channels))
        glorot_(self.lin.weight)

    @staticmethod
    def propagation(a: torch.Tensor) -> torch.Tensor:
        a = a.clone()
        idx = torch.arange(a.shape[0])
        a[idx, idx] = torch.where(a[idx, idx] > 0, a[idx, idx], torch.ones_like(a[idx, idx]))  # remaining self-loops
        d = a.sum(0).pow(-0.5)                                # in-degree (PyG normalises on the target)
        return (d[:, None] * a * d[None, :]).T                # out[dst] = sum_src M[dst, src] x[src]

    def forward(self, x: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
        return m @ self.lin(x) + self.bias


class TGCN(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.out_channels = out_channels
        self.conv_z = GCNConv(in_channels, out_channels)
        self.linear_z = nn.Linear(2 * out_channels, out_channels)
        self.conv_r = GCNConv(in_channels, out_channels)
        self.linear_r = nn.Linear(2 * out_channels, out_channels)
        self.conv_h = GCNConv(in_channels, out_channels)
        self.linear_h = nn.Linear(2 * out_channels, out_channels)

    def forward(self, x: torch.Tensor, m: torch.Tensor, h: torch.Tensor | None = None) -> torch.Tensor:
        if h is None:
            h = torch.zeros(*x.shape[:-1], self.out_channels)
        z = torch.sigmoid(self.linear_z(torch.cat([self.conv_z(x, m), h], -1)))
        r = torch.sigmoid(self.linear_r(torch.cat([self.conv_r(x, m), h], -1)))
        h_tilde = torch.tanh(self.linear_h(torch.cat([self.conv_h(x, m), h * r], -1)))
        return z * h + (1 - z) * h_tilde


class A3TGCNCore(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, periods: int):
        super().__init__()
        self.periods = periods
        self._base_tgcn = TGCN(in_channels, out_channels)
        self._attention = nn.Parameter(torch.empty(periods))
        nn.init.uniform_(self._attention)

    def forward(self, x: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
        """``x`` (B, N, in_channels, periods)."""
        probs = torch.softmax(self._attention, dim=0)
        return sum(probs[p] * self._base_tgcn(x[..., p], m, None) for p in range(self.periods))


class A3TGCN(nn.Module):
    def __init__(self, n_nodes: int, window: int, out: int, edge_index: torch.Tensor, hidden: int = 32):
        super().__init__()
        self.core = A3TGCNCore(1, hidden, window)
        self.head = Head(hidden, out)
        self.register_buffer("m", GCNConv.propagation(dense_adjacency(edge_index, n_nodes)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(torch.relu(self.core(x.unsqueeze(2), self.m)))

### ASTGCN — spatial and temporal attention around a Chebyshev graph convolution

Two blocks. Each computes temporal attention $E$ (a softmax over week pairs) and
spatial attention $S$ (a softmax over district pairs), then a K = 3 Chebyshev
convolution on the scaled Laplacian $\tilde{L} = 2L/\lambda_{\max} - I$ with
$L = D_{\text{out}} - A$, where the first-order term is modulated elementwise by
$S$, followed by a (1×3) convolution over weeks, a residual connection and layer
norm. A final convolution over weeks produces the 16-channel representation.

In [ ]:
class ChebConvAttention(nn.Module):
    """``torch_geometric_temporal`` ``ChebConvAttention`` with ``normalization=None``, dense."""

    def __init__(self, in_channels: int, out_channels: int, K: int):
        super().__init__()
        self._weight = nn.Parameter(torch.empty(K, in_channels, out_channels))
        self._bias = nn.Parameter(torch.empty(out_channels))
        nn.init.xavier_uniform_(self._weight)
        nn.init.uniform_(self._bias)

    @staticmethod
    def scaled_laplacian(a: torch.Tensor) -> torch.Tensor:
        import numpy as np
        a = a.clone()
        a.fill_diagonal_(0.0)                                   # remove_self_loops
        lap = torch.diag(a.sum(1)) - a                          # get_laplacian(None): D_out - A
        ev = np.linalg.eigvals(lap.double().numpy())            # LaplacianLambdaMax: largest |eig|
        lam = float(ev[np.argmax(np.abs(ev))].real)
        return 2.0 * lap / lam - torch.eye(a.shape[0])

    def forward(self, x: torch.Tensor, lt: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
        n = x.shape[1]
        tx0 = torch.matmul((torch.eye(n) * s).permute(0, 2, 1), x)
        out = torch.matmul(tx0, self._weight[0])
        if self._weight.size(0) > 1:
            tx1 = torch.matmul(lt * s, tx0)                     # attention on the first order only
            out = out + torch.matmul(tx1, self._weight[1])
        for k in range(2, self._weight.size(0)):
            tx2 = 2.0 * torch.matmul(lt, tx1) - tx0
            out = out + torch.matmul(tx2, self._weight[k])
            tx0, tx1 = tx1, tx2
        return out + self._bias


class SpatialAttention(nn.Module):
    def __init__(self, in_channels: int, num_of_vertices: int, num_of_timesteps: int):
        super().__init__()
        self._W1 = nn.Parameter(torch.FloatTensor(num_of_timesteps))
        self._W2 = nn.Parameter(torch.FloatTensor(in_channels, num_of_timesteps))
        self._W3 = nn.Parameter(torch.FloatTensor(in_channels))
        self._bs = nn.Parameter(torch.FloatTensor(1, num_of_vertices, num_of_vertices))
        self._Vs = nn.Parameter(torch.FloatTensor(num_of_vertices, num_of_vertices))
        for p in self.parameters():
            nn.init.xavier_uniform_(p) if p.dim() > 1 else nn.init.uniform_(p)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lhs = torch.matmul(torch.matmul(x, self._W1), self._W2)
        rhs = torch.matmul(self._W3, x).transpose(-1, -2)
        s = torch.matmul(self._Vs, torch.sigmoid(torch.matmul(lhs, rhs) + self._bs))
        return F.softmax(s, dim=1)


class TemporalAttention(nn.Module):
    def __init__(self, in_channels: int, num_of_vertices: int, num_of_timesteps: int):
        super().__init__()
        self._U1 = nn.Parameter(torch.FloatTensor(num_of_vertices))
        self._U2 = nn.Parameter(torch.FloatTensor(in_channels, num_of_vertices))
        self._U3 = nn.Parameter(torch.FloatTensor(in_channels))
        self._be = nn.Parameter(torch.FloatTensor(1, num_of_timesteps, num_of_timesteps))
        self._Ve = nn.Parameter(torch.FloatTensor(num_of_timesteps, num_of_timesteps))
        for p in self.parameters():
            nn.init.xavier_uniform_(p) if p.dim() > 1 else nn.init.uniform_(p)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lhs = torch.matmul(torch.matmul(x.permute(0, 3, 2, 1), self._U1), self._U2)
        rhs = torch.matmul(self._U3, x)
        e = torch.matmul(self._Ve, torch.sigmoid(torch.matmul(lhs, rhs) + self._be))
        return F.softmax(e, dim=1)


class ASTGCNBlock(nn.Module):
    def __init__(self, in_channels: int, K: int, nb_chev_filter: int, nb_time_filter: int,
                 time_strides: int, num_of_vertices: int, num_of_timesteps: int):
        super().__init__()
        self._temporal_attention = TemporalAttention(in_channels, num_of_vertices, num_of_timesteps)
        self._spatial_attention = SpatialAttention(in_channels, num_of_vertices, num_of_timesteps)
        self._chebconv_attention = ChebConvAttention(in_channels, nb_chev_filter, K)
        self._time_convolution = nn.Conv2d(nb_chev_filter, nb_time_filter, kernel_size=(1, 3),
                                           stride=(1, time_strides), padding=(0, 1))
        self._residual_convolution = nn.Conv2d(in_channels, nb_time_filter, kernel_size=(1, 1),
                                               stride=(1, time_strides))
        self._layer_norm = nn.LayerNorm(nb_time_filter)
        for p in self.parameters():
            nn.init.xavier_uniform_(p) if p.dim() > 1 else nn.init.uniform_(p)

    def forward(self, x: torch.Tensor, lt: torch.Tensor) -> torch.Tensor:
        b, n, f, t = x.shape
        x_tilde = torch.matmul(x.reshape(b, -1, t), self._temporal_attention(x)).reshape(b, n, f, t)
        s = self._spatial_attention(x_tilde)
        x_hat = F.relu(torch.cat([self._chebconv_attention(x[:, :, :, k], lt, s).unsqueeze(-1)
                                  for k in range(t)], dim=-1))
        x_hat = self._time_convolution(x_hat.permute(0, 2, 1, 3))
        x = self._residual_convolution(x.permute(0, 2, 1, 3))
        x = self._layer_norm(F.relu(x + x_hat).permute(0, 3, 2, 1))
        return x.permute(0, 2, 3, 1)


class ASTGCNCore(nn.Module):
    def __init__(self, nb_block: int, in_channels: int, K: int, nb_chev_filter: int, nb_time_filter: int,
                 time_strides: int, num_for_predict: int, len_input: int, num_of_vertices: int):
        super().__init__()
        self._blocklist = nn.ModuleList(
            [ASTGCNBlock(in_channels, K, nb_chev_filter, nb_time_filter, time_strides,
                         num_of_vertices, len_input)]
            + [ASTGCNBlock(nb_time_filter, K, nb_chev_filter, nb_time_filter, 1, num_of_vertices,
                           len_input // time_strides) for _ in range(nb_block - 1)])
        self._final_conv = nn.Conv2d(int(len_input / time_strides), num_for_predict,
                                     kernel_size=(1, nb_time_filter))
        for p in self.parameters():
            nn.init.xavier_uniform_(p) if p.dim() > 1 else nn.init.uniform_(p)

    def forward(self, x: torch.Tensor, lt: torch.Tensor) -> torch.Tensor:
        for block in self._blocklist:
            x = block(x, lt)
        x = self._final_conv(x.permute(0, 3, 1, 2))[:, :, :, -1]
        return x.permute(0, 2, 1)


class ASTGCN(nn.Module):
    def __init__(self, n_nodes: int, window: int, out: int, edge_index: torch.Tensor, feat: int = 16):
        super().__init__()
        # The reference sets num_for_predict = horizon, making the last convolution the
        # forecast; widening it to `feat` and projecting gives the same seam as the others.
        self.core = ASTGCNCore(nb_block=2, in_channels=1, K=3, nb_chev_filter=64, nb_time_filter=64,
                               time_strides=1, num_for_predict=feat, len_input=window,
                               num_of_vertices=n_nodes)
        self.head = Head(feat, out)
        self.register_buffer("lt", ChebConvAttention.scaled_laplacian(dense_adjacency(edge_index, n_nodes)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.core(x.unsqueeze(2), self.lt))

### DCRNN — a diffusion-convolutional GRU

The diffusion convolution sums $K = 32$ powers of the forward and reverse random-walk
transition matrices applied to the input, each with its own weights. The GRU gates
are diffusion convolutions of the input concatenated with the state. The benchmark
feeds the 3-week window as input channels and takes one GRU step from a zero state.

In [ ]:
class DConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, K: int):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(2, K, in_channels, out_channels))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.xavier_uniform_(self.weight)
        nn.init.zeros_(self.bias)

    @staticmethod
    def transitions(edge_index: torch.Tensor, n: int) -> tuple[torch.Tensor, torch.Tensor]:
        """Dense forms of the reference's two message-passing operators.

        Forward: each edge (s, d) carries x[s] / deg_out[s] into d.
        Reverse: the reference builds the reversed edge list from the transposed
        dense adjacency (row-major order) but indexes its weights with the ORIGINAL
        list's source nodes, position by position. That pairing is reproduced as is.
        """
        a = dense_adjacency(edge_index, n)
        deg_out, deg_in = a.sum(1), a.sum(0)
        src, dst = torch.nonzero(a, as_tuple=True)              # row-major, as the reference's list
        p_out = torch.zeros(n, n)
        p_out[dst, src] = 1.0 / deg_out[src]
        rsrc, rdst = torch.nonzero(a.T, as_tuple=True)          # dense_to_sparse(A^T), row-major
        w_in = 1.0 / deg_in[src]                                # norm_in = deg_in_inv[row], by position
        p_in = torch.zeros(n, n)
        p_in.index_put_((rdst, rsrc), w_in, accumulate=True)
        return p_out, p_in

    def forward(self, x: torch.Tensor, p_out: torch.Tensor, p_in: torch.Tensor) -> torch.Tensor:
        tx0 = tx1 = x
        h = torch.matmul(tx0, self.weight[0][0]) + torch.matmul(tx0, self.weight[1][0])
        if self.weight.size(1) > 1:
            tx1_o, tx1_i = p_out @ x, p_in @ x
            h = h + torch.matmul(tx1_o, self.weight[0][1]) + torch.matmul(tx1_i, self.weight[1][1])
        for k in range(2, self.weight.size(1)):
            tx2_o = 2.0 * (p_out @ tx1_o) - tx0
            tx2_i = 2.0 * (p_in @ tx1_i) - tx0
            h = h + torch.matmul(tx2_o, self.weight[0][k]) + torch.matmul(tx2_i, self.weight[1][k])
            tx0, tx1_o, tx1_i = tx1, tx2_o, tx2_i                # tx1 is never updated: tx0 stays x
        return h + self.bias


class DCRNNCell(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, K: int):
        super().__init__()
        self.out_channels = out_channels
        self.conv_x_z = DConv(in_channels + out_channels, out_channels, K)
        self.conv_x_r = DConv(in_channels + out_channels, out_channels, K)
        self.conv_x_h = DConv(in_channels + out_channels, out_channels, K)

    def forward(self, x, p_out, p_in, h=None):
        if h is None:
            h = torch.zeros(*x.shape[:-1], self.out_channels)
        z = torch.sigmoid(self.conv_x_z(torch.cat([x, h], -1), p_out, p_in))
        r = torch.sigmoid(self.conv_x_r(torch.cat([x, h], -1), p_out, p_in))
        h_tilde = torch.tanh(self.conv_x_h(torch.cat([x, h * r], -1), p_out, p_in))
        return z * h + (1 - z) * h_tilde


class DCRNN(nn.Module):
    def __init__(self, n_nodes: int, window: int, out: int, edge_index: torch.Tensor,
                 hidden: int = 64, K: int = 32):
        super().__init__()
        self.core = DCRNNCell(window, hidden, K)
        self.head = Head(hidden, out)
        p_out, p_in = DConv.transitions(edge_index, n_nodes)
        self.register_buffer("p_out", p_out)
        self.register_buffer("p_in", p_in)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(torch.relu(self.core(x, self.p_out, self.p_in)))

### AAGCN — attention-augmented graph convolution with a temporal convolution

Three graph partitions (self, inward, outward — column-normalised adjacency and its
transpose), a 1×1 convolution per partition, batch norm and a residual; then
spatial, temporal and channel attention; then a (9×1) temporal convolution. The
benchmark runs it non-adaptive (the learned-adjacency branch off) with 8 channels.

In [ ]:
class UnitTCN(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 9, stride: int = 1):
        super().__init__()
        pad = int((kernel_size - 1) / 2)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=(kernel_size, 1),
                              padding=(pad, 0), stride=(stride, 1))
        self.bn = nn.BatchNorm2d(out_channels)
        nn.init.kaiming_normal_(self.conv.weight, mode="fan_out")
        nn.init.constant_(self.conv.bias, 0)
        nn.init.constant_(self.bn.weight, 1)
        nn.init.constant_(self.bn.bias, 0)

    def forward(self, x):
        return self.bn(self.conv(x))


class UnitGCN(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, A: torch.Tensor, coff_embedding: int = 4,
                 num_subset: int = 3):
        super().__init__()
        self.out_c, self.num_subset = out_channels, num_subset
        self.register_buffer("A", A)
        num_jpts = A.shape[-1]
        self.conv_d = nn.ModuleList(nn.Conv2d(in_channels, out_channels, 1) for _ in range(num_subset))
        self.conv_ta = nn.Conv1d(out_channels, 1, 9, padding=4)
        ker_jpt = num_jpts - 1 if not num_jpts % 2 else num_jpts
        self.conv_sa = nn.Conv1d(out_channels, 1, ker_jpt, padding=(ker_jpt - 1) // 2)
        self.fc1c = nn.Linear(out_channels, out_channels // 2)
        self.fc2c = nn.Linear(out_channels // 2, out_channels)
        self.down = (nn.Sequential(nn.Conv2d(in_channels, out_channels, 1), nn.BatchNorm2d(out_channels))
                     if in_channels != out_channels else nn.Identity())
        self.bn = nn.BatchNorm2d(out_channels)
        # Initialisation exactly as the reference, in its order.
        nn.init.constant_(self.conv_ta.weight, 0)
        nn.init.constant_(self.conv_ta.bias, 0)
        nn.init.xavier_normal_(self.conv_sa.weight)
        nn.init.constant_(self.conv_sa.bias, 0)
        nn.init.kaiming_normal_(self.fc1c.weight)
        nn.init.constant_(self.fc1c.bias, 0)
        nn.init.constant_(self.fc2c.weight, 0)
        nn.init.constant_(self.fc2c.bias, 0)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        nn.init.constant_(self.bn.weight, 1e-6)
        nn.init.constant_(self.bn.bias, 0)
        for conv in self.conv_d:
            n, k1, k2 = conv.weight.size(0), conv.weight.size(1), conv.weight.size(2)
            nn.init.normal_(conv.weight, 0, math.sqrt(2.0 / (n * k1 * k2 * num_subset)))
            nn.init.constant_(conv.bias, 0)

    def forward(self, x):
        n, c, t, v = x.size()
        y = None
        for i in range(self.num_subset):
            z = self.conv_d[i](torch.matmul(x.reshape(n, c * t, v), self.A[i]).view(n, c, t, v))
            y = z if y is None else z + y
        y = F.relu(self.bn(y) + self.down(x))
        se = torch.sigmoid(self.conv_sa(y.mean(-2)))                           # spatial attention
        y = y * se.unsqueeze(-2) + y
        se = torch.sigmoid(self.conv_ta(y.mean(-1)))                           # temporal attention
        y = y * se.unsqueeze(-1) + y
        se = torch.sigmoid(self.fc2c(F.relu(self.fc1c(y.mean(-1).mean(-1)))))  # channel attention
        return y * se.unsqueeze(-1).unsqueeze(-1) + y


class AAGCNCore(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, A: torch.Tensor):
        super().__init__()
        self.gcn1 = UnitGCN(in_channels, out_channels, A)
        self.tcn1 = UnitTCN(out_channels, out_channels)
        self.residual = UnitTCN(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return F.relu(self.tcn1(self.gcn1(x)) + self.residual(x))


class AAGCN(nn.Module):
    def __init__(self, n_nodes: int, window: int, out: int, edge_index: torch.Tensor, channels: int = 8):
        super().__init__()
        a = dense_adjacency(edge_index, n_nodes)
        inward = F.normalize(a, dim=0, p=1)
        outward = F.normalize(a.T, dim=0, p=1)
        self.core = AAGCNCore(1, channels, torch.stack([torch.eye(n_nodes), inward, outward]))
        self.head = Head(channels * window, out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        o = self.core(x.permute(0, 2, 1).unsqueeze(1))                         # (B, C, T, N)
        b, c, t, n = o.shape
        return self.head(o.permute(0, 3, 1, 2).reshape(b, n, c * t))

### LSTM — the SEIR-LSTM's encoder, no graph at all

Two LSTM layers over each district's own 3-week window. It is the control that says
what the graph adds.

In [ ]:
class LSTMEncoder(nn.Module):
    def __init__(self, n_nodes: int, window: int, out: int, edge_index: torch.Tensor | None = None,
                 layers: int = 2):
        super().__init__()
        self.lstm = nn.LSTM(1, out, num_layers=layers, batch_first=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n, w = x.shape
        h, _ = self.lstm(x.reshape(b * n, w, 1))
        return h[:, -1].reshape(b, n, -1)


ENCODERS = {"STGAT": STGAT, "A3TGCN": A3TGCN, "ASTGCN": ASTGCN, "DCRNN": DCRNN, "AAGCN": AAGCN,
            "LSTM": LSTMEncoder}


class Encoder(nn.Module):
    """One of the six as the representation for the heads.

    The case window goes through the architecture at its reference contract; any
    exogenous channels (seasonal features, climate) go around it and are joined at
    the head, so the published architecture runs exactly as designed.
    """

    def __init__(self, name: str, n_nodes: int, window: int, extra: int, hidden: int,
                 edge_index: torch.Tensor):
        super().__init__()
        self.window, self.extra = window, extra
        self.core = ENCODERS[name](n_nodes, window, hidden, edge_index)
        self.out_dim = hidden + extra

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = torch.relu(self.core(x[..., : self.window].contiguous()))
        return torch.cat([h, x[..., self.window:]], dim=-1) if self.extra else h

## 5. The SEIR model and the forecast heads

### 5.1 A differentiable SEIR simulator

$\dot S = -\lambda S,\ \dot E = \lambda S - \omega E,\ \dot I = \omega E - \gamma I,\ \dot R = \gamma I$,
stepped **daily** (7 sub-steps per week) with exponential flows — each step moves
$S(1-e^{-\lambda\,dt})$ out of S, and so on — which is the exact outflow for rates held
over a step and keeps every compartment non-negative. Rates per day: incubation
$\omega = 0.1$, recovery $\gamma = 1/7$. Weekly incidence is the flow E → I over the
week. The metapopulation variant recomputes the force of infection every day as
$\beta_i \sum_j C_{ij} I_j / N_j$.

In [ ]:
DAYS_PER_WEEK = 7
OMEGA, GAMMA, RHO = 0.7 / 7.0, 1.0 / 7.0, 1.0 / 11.0


def seir_step(state, lam, omega, gamma, dt):
    s, e, i, r = state.unbind(-1)
    infect = s * (1.0 - torch.exp(-lam * dt))
    onset = e * (1.0 - torch.exp(torch.as_tensor(-omega * dt, dtype=state.dtype)))
    recover = i * (1.0 - torch.exp(torch.as_tensor(-gamma * dt, dtype=state.dtype)))
    return torch.stack([s - infect, e + infect - onset, i + onset - recover, r + recover], -1), onset


def simulate_weeks(state0, foi, omega=OMEGA, gamma=GAMMA, substeps=7):
    """One force of infection per district per week; returns (states, weekly incidence)."""
    dt = DAYS_PER_WEEK / substeps
    states, weekly, state = [state0], [], state0
    for w in range(foi.shape[-1]):
        lam, total = foi[..., w], torch.zeros_like(foi[..., w])
        for _ in range(substeps):
            state, onset = seir_step(state, lam, omega, gamma, dt)
            total = total + onset
        states.append(state)
        weekly.append(total)
    return torch.stack(states, -2), torch.stack(weekly, -1)


def simulate_closed_loop(state0, beta, omega=OMEGA, gamma=GAMMA, substeps=7, coupling=None):
    """Mass action, force of infection recomputed every day (optionally coupled across districts)."""
    dt = DAYS_PER_WEEK / substeps
    state, weekly = state0, []
    for w in range(beta.shape[-1]):
        total = torch.zeros(state.shape[:-1], dtype=state.dtype)
        for _ in range(substeps):
            prevalence = state[..., 2] / state.sum(-1).clamp_min(1.0)
            if coupling is not None:
                prevalence = prevalence @ coupling.T
            state, onset = seir_step(state, beta[..., w] * prevalence, omega, gamma, dt)
            total = total + onset
        weekly.append(total)
    return None, torch.stack(weekly, -1), None


def seir_state(cases_raw, pop, cum, rho=RHO, s0=1.0 - 0.682):
    """Initial (S, E, I, R) fractions from observed counts: I0 from cases[t-1], E0 from cases[t-2].

    Reporting covers a fraction rho of infections; S starts at 1 - 0.682 (a 2013-14 Colombo
    serosurvey, before the study period) less cumulative infections implied by reported cases.
    """
    scale = rho * pop
    i0 = (cases_raw[..., -1] / scale).clamp(1e-6, 0.5)
    e0 = (cases_raw[..., -2] / scale).clamp(1e-6, 0.5)
    s = (s0 - cum / scale).clamp(0.01, 1.0)
    return torch.stack([s, e0, i0, (1.0 - s - e0 - i0).clamp(0.0, 1.0)], -1)

### 5.2 The network: encoder → head

Every model is an encoder (section 4, or the toy graph layers used by the first
screen) followed by one of six heads. All heads output fold-standardised
$\log(1+y)$ for the three horizon weeks; $p$ is last week's value in the same space.

| head | forecast | role |
|---|---|---|
| `direct` | $f(h)$ | the published GNNs |
| `residual` | $p + f(h)$ | persistence anchor |
| `gated` | $p + \sigma(a)\,(f(h) - p)$ | anchor + learned gate, **no physics** |
| `foi` | SEIR$(\lambda = e^{f(h)+b})$ | the pure SEIR decoder |
| `foi_res` | $p + \sigma(a)\,(\mathrm{SEIR}(\lambda) - p)$ | gated SEIR: our SEIR-GNN, or SEIR-LSTM with the LSTM |
| `foi_meta` | as `foi_res`, metapopulation SEIR with learned coupling | physics inside the dynamics |

Options (all off unless an experiment sets them): negative-binomial output
(`dist="nb"`), reversible instance normalisation (`norm`), a district identity
embedding (`node_emb`), a nonlinear head (`head_mlp`), district seasonal curves
(`dseason`), a national context vector (`global_ctx`) and an auxiliary SEIR loss
(`aux_phys`).

In [ ]:
LAM_LOG0 = -9.36          # log of the median force of infection that reproduces observed counts
LOG_LAMBDA_MAX = -1.95    # log(1/7), a hard clamp
META_LOG_BETA0 = -0.69    # log(0.5/day): the metapopulation transmission rate at initialisation
N_SEASON = 4


class GraphLayer(nn.Module):
    """One message-passing step: `none` (identity), `gcn` (fixed graph), `adaptive` (learned), `hybrid`."""

    def __init__(self, dim, n_nodes, mode, emb=16):
        super().__init__()
        self.mode = mode
        self.lin = nn.Linear(dim, dim)
        self.e1 = nn.Parameter(torch.randn(n_nodes, emb) * 0.05)
        self.e2 = nn.Parameter(torch.randn(n_nodes, emb) * 0.05)

    def forward(self, h, fixed):
        if self.mode == "none":
            a = torch.eye(fixed.shape[0])
        elif self.mode == "gcn":
            a = fixed
        else:
            learned = torch.softmax(torch.relu(self.e1 @ self.e2.T), -1)
            a = learned if self.mode == "adaptive" else 0.5 * fixed + 0.5 * learned
        return torch.relu(self.lin(a @ h))


class Net(nn.Module):
    def __init__(self, in_dim, n_nodes, horizon=3, hidden=64, backbone="gcn", head="residual", layers=2,
                 dropout=0.1, lam_param="sigmoid", state_fit=False, window=3, dist="point", norm="fold",
                 node_emb=0, aux_phys=0.0, head_mlp=0, dseason=False, global_ctx=False):
        super().__init__()
        self.head, self.backbone, self.dist, self.norm, self.window = head, backbone, dist, norm, window
        self.lam_param, self.state_fit, self.aux_phys, self.global_ctx = lam_param, state_fit, aux_phys, global_ctx
        if backbone in ENCODERS:
            self.real = Encoder(backbone, n_nodes, window, in_dim - window, hidden, EDGE_INDEX)
            self.in_proj, self.graph = None, nn.ModuleList()
            feat = self.real.out_dim
        elif backbone == "linear":
            self.real, self.in_proj, self.graph = None, None, nn.ModuleList()
            feat = in_dim
        else:
            self.real = None
            self.in_proj = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout))
            self.graph = nn.ModuleList(GraphLayer(hidden, n_nodes, backbone) for _ in range(layers))
            feat = hidden
        if global_ctx:
            feat *= 2
        self.node_emb = nn.Parameter(torch.randn(n_nodes, node_emb) * 0.1) if node_emb else None
        feat += node_emb
        if norm == "revin":
            self.rev_gamma = nn.Parameter(torch.ones(1))
            self.rev_beta = nn.Parameter(torch.zeros(1))
        self.drop = nn.Dropout(dropout)
        self.out = (nn.Sequential(nn.Linear(feat, head_mlp), nn.ReLU(), nn.Dropout(dropout), nn.Linear(head_mlp, horizon))
                    if head_mlp else nn.Linear(feat, horizon))
        self.disp = nn.Linear(feat, horizon) if dist == "nb" else None
        self.out_phys = nn.Linear(feat, horizon) if aux_phys else None
        if head == "gated":
            self.alpha = nn.Parameter(torch.tensor(-2.0))
        if head in ("foi", "foi_res", "foi_meta") or aux_phys:
            self.alpha = nn.Parameter(torch.tensor(-2.0))            # gate, through a sigmoid: starts at 0.12
            self.beta = nn.Parameter(torch.tensor(0.0))              # log-scale spatial import weight
            self.lam_bias = nn.Parameter(torch.tensor(LAM_LOG0))
            self.e_scale = nn.Parameter(torch.tensor(0.0))           # learned scale on E0
            self.rho_scale = nn.Parameter(torch.tensor(0.0))         # learned scale on reporting
        self.dseason = nn.Parameter(torch.zeros(n_nodes, N_SEASON, horizon)) if dseason else None
        if head == "foi_meta":
            self.beta_bias = nn.Parameter(torch.tensor(float(META_LOG_BETA0)))
            self.c_logit = nn.Parameter(torch.zeros(n_nodes, n_nodes))

    def encode(self, x, fixed):
        if self.real is not None:
            h = self.drop(self.real(x))
        elif self.backbone == "linear":
            h = x
        else:
            h = self.in_proj(x)
            for layer in self.graph:
                h = h + self.drop(layer(h, fixed))
        if self.global_ctx:
            h = torch.cat([h, h.mean(1, keepdim=True).expand_as(h)], -1)
        if self.node_emb is not None:
            h = torch.cat([h, self.node_emb.expand(h.shape[0], -1, -1)], -1)
        return h

    def _fit_state(self, st0):
        s, e, i, r = st0.unbind(-1)
        scaled = e * torch.sigmoid(self.e_scale) * 2.0
        return torch.stack([s, scaled, i, (r + e - scaled).clamp_min(0.0)], -1)

    def _physics(self, raw, st0, pop, mean, std):
        lam = (torch.exp((raw + self.lam_bias).clamp(-25.0, LOG_LAMBDA_MAX)) if self.lam_param == "log"
               else (1.0 / 7.0) * torch.sigmoid(raw))
        if self.state_fit:
            st0 = self._fit_state(st0)
        imp = st0[..., 2].clamp_min(1e-9).log().unsqueeze(-1)       # spatial import on the log infected fraction
        lam = lam * torch.exp(self.beta * (imp - imp.mean()) * 0.1)
        _, inc = simulate_weeks(st0, lam)
        rho = RHO * torch.exp(self.rho_scale) if self.state_fit else RHO
        return (torch.log1p((inc * rho * pop.unsqueeze(-1)).clamp_min(0.0)) - mean) / std

    def _meta_physics(self, raw, st0, pop, mean, std, fixed):
        beta = torch.exp((raw + self.beta_bias).clamp(-12.0, 2.0))
        if self.state_fit:
            st0 = self._fit_state(st0)
        coupling = torch.softmax(torch.log(fixed.clamp_min(1e-9)) + self.c_logit, -1)
        _, inc, _ = simulate_closed_loop(st0, beta, coupling=coupling)
        rho = RHO * torch.exp(self.rho_scale) if self.state_fit else RHO
        return (torch.log1p((inc * rho * pop.unsqueeze(-1)).clamp_min(0.0)) - mean) / std

    def forward(self, x, fixed, p_z, st0=None, pop=None, mean=0.0, std=1.0):
        season = x[..., -N_SEASON:] if self.dseason is not None else None
        if self.norm != "fold":
            cases = x[..., : self.window]
            m = cases.mean(-1, keepdim=True)
            s = torch.sqrt(cases.var(-1, unbiased=False, keepdim=True) + 0.01)
            cases = cases - m
            if self.norm == "revin":
                cases = cases / s * self.rev_gamma + self.rev_beta
            x = torch.cat([cases, x[..., self.window:]], -1)
        h = self.encode(x, fixed)
        raw = self.out(h)
        if self.dseason is not None:
            raw = raw + torch.einsum("bnk,nkh->bnh", season, self.dseason)
        disp = self.disp(h) if self.disp is not None else None
        aux = self._physics(self.out_phys(h), st0, pop, mean, std) if self.out_phys is not None else None
        if self.head == "direct":
            if self.norm == "revin_mean":
                raw = raw + m
            elif self.norm == "revin":
                raw = (raw - self.rev_beta) / (self.rev_gamma.abs() + 1e-3) * s + m
            return raw, disp, aux
        if self.head == "residual":
            return p_z + raw, disp, aux
        if self.head == "gated":
            return p_z + torch.sigmoid(self.alpha) * (raw - p_z), disp, aux
        if self.head == "foi_meta":
            phys = self._meta_physics(raw, st0, pop, mean, std, fixed)
            return p_z + torch.sigmoid(self.alpha) * (phys - p_z), disp, aux
        phys = self._physics(raw, st0, pop, mean, std)
        if self.head == "foi":
            return phys, disp, aux
        return p_z + torch.sigmoid(self.alpha) * (phys - p_z), disp, aux

### 5.3 Losses

Squared error on the standardised $\log(1+y)$ (`mse_z`), or the **negative-binomial**
likelihood (`nb`): with $\mu = e^{\hat z \sigma + m} - 1$ and a learned dispersion
$\alpha \in [10^{-4}, 2]$, it minimises
$-\log \mathrm{NB2}(y; \mu, \alpha)$. Squared error on $\log(1+y)$ makes
$e^{\hat z}$ a conditional *median*, while RMSE on counts rewards the conditional
*mean*; the NB's $\mu$ is the mean. The spatial physics penalty adds
$\lambda_s \sum_{(i,j)} A_{ij}(\hat y_i/s_i - \hat y_j/s_j)^2$ over bordering districts,
with $s_i$ a district's training-mean cases (or the same in $\log(1+\cdot)$).

In [ ]:
ALPHA_MIN, ALPHA_MAX, LOG_MU_MAX = 1e-4, 2.0, 12.0


def nb_nll(mu, alpha, target, weight=None):
    y, r = target.clamp_min(0.0), 1.0 / alpha
    log_r_mu = torch.log(r + mu)
    nll = -(torch.lgamma(y + r) - torch.lgamma(r) - torch.lgamma(y + 1.0)
            + r * (torch.log(r) - log_r_mu) + y * (torch.log(mu) - log_r_mu))
    return nll.mean() if weight is None else (nll * weight).sum() / weight.sum()


def loss_fn(kind, pred_z, batch, mean, std, disp=None):
    if kind == "nb":
        mu = torch.expm1(torch.clamp(pred_z * std + mean, -1.0, LOG_MU_MAX)).clamp_min(1e-6)
        alpha = ALPHA_MIN + (ALPHA_MAX - ALPHA_MIN) * torch.sigmoid(disp)
        return nb_nll(mu, alpha, batch["y_raw"], batch.get("w"))
    if kind == "mse_z":
        return F.mse_loss(pred_z, batch["y_z"])
    raise ValueError(kind)


def spatial_penalty(counts, adj, scales, kind):
    n = counts.shape[1]
    if kind == "ratio":
        rel = counts / scales.view(1, n, 1).clamp_min(1.0)
    else:
        rel = torch.log1p(counts.clamp_min(0.0)) - torch.log1p(scales.clamp_min(0.0)).view(1, n, 1)
    diff = rel.unsqueeze(2) - rel.unsqueeze(1)
    weighted = adj.unsqueeze(0).unsqueeze(-1) * diff.pow(2)
    return weighted.sum() / (weighted.shape[0] * weighted.shape[-1] * adj.abs().sum().clamp_min(1e-8))


def to_counts(pred_z, mean, std):
    return torch.expm1(torch.clamp(pred_z * std + mean, -1.0, 12.0)).clamp_min(0.0).detach().numpy()

### 5.4 Why a pure SEIR decoder cannot work here (measured before any training)

For a given initial state, the simulator's weekly incidence rises monotonically with
the force of infection λ, so λ can be **inverted by bisection** for the value that
reproduces each observed count exactly. That answers two questions without training
anything:

* **Reach.** With transmission switched off (λ = 0), the exposed people already seeded
  from recent cases still produce some cases. Any target below that floor is
  unreachable at any network output.
* **Learnability.** How much of the required λ (in logs) is predictable from last
  week's cases, compared with how predictable the target itself is.

In [ ]:
def incidence(state, lam):
    _, inc = simulate_weeks(state, lam.unsqueeze(-1))
    return inc[..., 0]


def advance(state, lam):
    st, _ = simulate_weeks(state, lam.unsqueeze(-1))
    return st[..., 1, :]


def invert(state, pop, target, iters=60):
    lo, hi = torch.zeros_like(target), torch.full_like(target, 10.0 / 7.0)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        low = incidence(state, mid) * (RHO * pop) < target
        lo, hi = torch.where(low, mid, lo), torch.where(low, hi, mid)
    return 0.5 * (lo + hi)


def r2(x, y):
    ok = np.isfinite(x) & np.isfinite(y)
    a = np.stack([x[ok], np.ones(ok.sum())], 1)
    res = y[ok] - a @ np.linalg.lstsq(a, y[ok], rcond=None)[0]
    return float(1.0 - res.var() / y[ok].var())


def cumulative_before(idx):
    c = np.nan_to_num(CASES)
    return torch.tensor(np.stack([c[:i].sum(0) for i in idx]), dtype=torch.float32)


diag = []
for fold in build_folds():
    pack = build_tensors(fold, "train")
    st = seir_state(pack["x_raw"], pack["pop"], cumulative_before(pack["idx"]))
    y, pop = pack["y_raw"], pack["pop"]
    floor, s_lo, lam_star, s_hi = [], st, [], st
    for h in range(HORIZON):
        z = torch.zeros(st.shape[:-1])
        floor.append(incidence(s_lo, z) * (RHO * pop))
        s_lo = advance(s_lo, z)
        lam = invert(s_hi, pop, y[..., h])
        lam_star.append(lam)
        s_hi = advance(s_hi, lam)
    floor, lam_star = torch.stack(floor, -1), torch.stack(lam_star, -1)
    last = pack["x_raw"][..., -1].numpy()
    lg = np.log(np.clip(lam_star[..., 0].numpy(), 1e-8, None))
    diag.append({"origin": fold.origin,
                 "targets below the lambda=0 floor (%)": 100 * float((y < floor).float().mean()),
                 "cells needing lambda = 0 (%)": 100 * float((lam_star <= 1e-6 / 7).float().mean()),
                 "r2 of log lambda* on log cases[t-1]": r2(np.log1p(last).ravel(), lg.ravel()),
                 "r2 of log cases[t] on log cases[t-1]": r2(np.log1p(last).ravel(),
                                                             np.log1p(y[..., 0].numpy()).ravel()),
                 "S at its 0.01 clamp (%)": 100 * float((st[..., 0] <= 0.0100001).float().mean())})
DIAG = pd.DataFrame(diag).set_index("origin")
print(DIAG.round(3).to_string())

## 6. Training

One run = one configuration × one fold × one seed. Adam (learning rate 3e-3, weight
decay 1e-4, batch 32, cosine schedule), gradient clipping at 5, early stopping on
**validation RMSE on counts** with patience 40, and the best-validation weights
restored. Test is scored once, at the end, and never used to choose anything.
Every run is single-threaded, so a configuration is reproducible from its seed.

In [ ]:
def cumulative(idx):
    return cumulative_before(idx)


def run_fold(fold, *, backbone, head, loss="mse_z", use_climate=False, use_season=False, seed=0,
             hidden=64, layers=2, dropout=0.1, lr=3e-3, weight_decay=1e-4, epochs=300, batch_size=32,
             patience=40, lam_param="sigmoid", state_fit=False, dist="point", norm="fold", node_emb=0,
             aux_phys=0.0, train_frac=1.0, augment="none", clim_blocks=(), head_mlp=0, dseason=False,
             global_ctx=False, spatial=0.0, spatial_kind="ratio", keep_train=False):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if train_frac < 1.0:
        tr = fold.idx["train"]
        sub = np.sort(np.random.default_rng(1000 + seed).choice(tr, max(1, int(round(train_frac * len(tr)))),
                                                               replace=False))
        fold = dc_replace(fold, idx={**fold.idx, "train": sub})
    if clim_blocks:
        fold = with_history(fold, max(b for _, b in clim_blocks))
    packs = {s: build_tensors(fold, s, use_climate, use_season, clim_blocks) for s in ("train", "val", "test")}
    needs_state = head in ("foi", "foi_res", "foi_meta") or aux_phys > 0
    pretrain = None
    n_real = len(packs["train"]["idx"])
    tail = {}
    if augment == "timegan":
        syn = synth_timegan(fold, n_real, seed)
        tail = {"synthetic_max": float(torch.cat([syn["x_raw"].flatten(), syn["y_raw"].flatten()]).max()),
                "real_train_max": float(torch.cat([packs["train"]["x_raw"].flatten(),
                                                   packs["train"]["y_raw"].flatten()]).max())}
        packs["train"] = concat_packs(packs["train"], syn)
    elif augment == "seir_pretrain_cal":
        pretrain = synth_seir(fold, n_real, seed)
    state = {s: seir_state(p["x_raw"], p["pop"], cumulative(p["idx"])) if needs_state else None
             for s, p in packs.items()}
    net = Net(packs["train"]["x"].shape[-1], N, horizon=HORIZON, hidden=hidden, backbone=backbone, head=head,
              layers=layers, dropout=dropout, lam_param=lam_param, state_fit=state_fit, window=fold.window,
              dist=dist, norm=norm, node_emb=node_emb, aux_phys=aux_phys, head_mlp=head_mlp, dseason=dseason,
              global_ctx=global_ctx)
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    def predict(split):
        p = packs[split]
        return net(p["x"], FIXED, p["p_z"], state[split], p["pop"], fold.mean, fold.std)[0]

    if pretrain is not None:           # learn from simulated epidemics first; no early stopping here
        m = len(pretrain["idx"])
        for _ in range(50):
            net.train()
            for sl in torch.randperm(m).split(batch_size):
                b = {k: v[sl] for k, v in pretrain.items() if k != "idx"}
                opt.zero_grad()
                pred, disp, _ = net(b["x"], FIXED, b["p_z"], None, b["pop"], fold.mean, fold.std)
                loss_fn(loss, pred, b, fold.mean, fold.std, disp).backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
                opt.step()
        opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=weight_decay)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    if spatial:
        adj_bin = ((FIXED > 0) & ~torch.eye(N, dtype=torch.bool)).float()
        scales = torch.tensor(np.nanmean(CASES[: int(fold.idx["train"].max())], axis=0), dtype=torch.float32)

    best, best_state, waited, ran = float("inf"), None, 0, 0
    n = len(packs["train"]["idx"])
    for ep in range(epochs):
        ran = ep + 1
        net.train()
        for sl in torch.randperm(n).split(batch_size):
            batch = {k: v[sl] for k, v in packs["train"].items() if k != "idx"}
            st = state["train"][sl] if needs_state else None
            opt.zero_grad()
            pred, disp, aux = net(batch["x"], FIXED, batch["p_z"], st, batch["pop"], fold.mean, fold.std)
            out = loss_fn(loss, pred, batch, fold.mean, fold.std, disp)
            if spatial:
                counts = torch.expm1(torch.clamp(pred * fold.std + fold.mean, -1.0, 12.0)).clamp_min(0.0)
                out = out + spatial * spatial_penalty(counts, adj_bin, scales, spatial_kind)
            if aux is not None:
                out = out + aux_phys * F.mse_loss(aux, batch["y_z"])
            out.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            opt.step()
        sched.step()
        net.eval()
        with torch.no_grad():
            v = rmse(to_counts(predict("val"), fold.mean, fold.std), packs["val"]["y_raw"].numpy())
        if v < best - 1e-6:
            best, waited = v, 0
            best_state = {k: t.detach().clone() for k, t in net.state_dict().items()}
        else:
            waited += 1
            if waited >= patience:
                break
    net.load_state_dict(best_state)
    net.eval()
    splits = ("train", "val", "test") if keep_train else ("val", "test")
    with torch.no_grad():
        preds = {s: to_counts(predict(s), fold.mean, fold.std) for s in splits}
    truth = {s: packs[s]["y_raw"].numpy() for s in splits}
    row = score(preds["test"], truth["test"])
    row.update(val_RMSE=rmse(preds["val"], truth["val"]), best_epoch=ran - waited, epochs_ran=ran,
               n_train=len(fold.idx["train"]), **tail)
    return row, {"pred": preds, "truth": truth, "persist": {s: packs[s]["p_raw"].numpy() for s in splits},
                 "idx": {s: np.asarray(packs[s]["idx"]) for s in splits}}

### 6.1 Forecasters without a network

* **k-nearest-neighbour analogues**: each (district, window) is matched to the k most
  similar training windows by the shape of its recent log-cases, optionally its level
  and season; the forecast applies the neighbours' realised growth. k and the two
  feature weights are chosen on validation.
* **Gradient-boosted trees** on log-growth from last week, recent history, season,
  district identity and optionally climate lag blocks, one model per horizon week.

In [ ]:
from itertools import product

GRID_K, GRID_LEVEL, GRID_SEASON = (10, 20, 40, 80, 160, 320), (0.0, 0.5, 1.0), (0.0, 0.5)


def library_idx(fold):
    """Training origins whose whole target span ends before the first validation origin."""
    tr = fold.idx["train"]
    return tr[tr + HORIZON - 1 < int(fold.idx["val"].min())]


def _knn_windows(idx, window, with_target):
    c = np.nan_to_num(CASES)
    hist = np.stack([np.log1p(c[i - window: i]).T for i in idx])
    last = hist[..., -1]
    season = np.repeat(seasonal_features(idx)[:, None, :], N, 1)
    growth = (np.stack([np.log1p(c[i: i + HORIZON]).T for i in idx]) - last[..., None]) if with_target else None
    return hist, last, season, growth


def _knn_predict(lib, query, k, wl, ws):
    feats = lambda h, l, s: np.concatenate([h - l[..., None], wl * l[..., None], ws * s], -1)  # noqa: E731
    lf = feats(*lib[:3]).reshape(-1, lib[0].shape[-1] + 5)
    qf = feats(*query[:3]).reshape(-1, query[0].shape[-1] + 5)
    lg, q_last = lib[3].reshape(-1, HORIZON), query[1].reshape(-1)
    lsq, out = (lf ** 2).sum(1)[None, :], np.empty((len(qf), HORIZON))
    for a in range(0, len(qf), 1024):
        q = qf[a: a + 1024]
        d2 = (q ** 2).sum(1)[:, None] - 2 * q @ lf.T + lsq
        nn_idx = np.argpartition(d2, kth=min(k, d2.shape[1] - 1), axis=1)[:, :k]
        out[a: a + 1024] = np.expm1(q_last[a: a + 1024, None, None] + lg[nn_idx]).mean(1)
    return out.clip(min=0.0).reshape(*query[1].shape, -1)


def run_knn(fold, seed=0, **_):
    lib = _knn_windows(library_idx(fold), fold.window, True)
    q = {s: _knn_windows(fold.idx[s], fold.window, False) for s in ("val", "test")}
    truth = {s: np.stack([CASES[i: i + HORIZON].T for i in fold.idx[s]]) for s in q}
    best = min(((rmse(_knn_predict(lib, q["val"], k, wl, ws), truth["val"]), k, wl, ws)
                for k, wl, ws in product(GRID_K, GRID_LEVEL, GRID_SEASON)))
    preds = {s: _knn_predict(lib, q[s], *best[1:]) for s in q}
    row = score(preds["test"], truth["test"])
    row.update(val_RMSE=best[0])
    persist = {s: np.repeat(CASES[fold.idx[s] - 1][:, :, None], HORIZON, 2) for s in q}
    return row, {"pred": preds, "truth": truth, "persist": persist, "idx": {s: fold.idx[s] for s in q}}


def run_gbm(fold, seed=0, clim_blocks=(), **_):
    from sklearn.ensemble import HistGradientBoostingRegressor

    if clim_blocks:
        fold = with_history(fold, max(b for _, b in clim_blocks))
    lc = np.log1p(np.nan_to_num(CASES))

    def design(idx):
        last = lc[idx - 1]
        parts = [last[..., None], np.stack([lc[idx - k] - last for k in (2, 3)], -1),
                 np.repeat(seasonal_features(idx)[:, None, :], N, 1), np.repeat(np.eye(N)[None], len(idx), 0)]
        if clim_blocks:
            blk, ref = climate_blocks(idx, clim_blocks, CLIMATE), climate_blocks(fold.idx["train"], clim_blocks, CLIMATE)
            parts.append((blk - ref.mean((0, 1))) / (ref.std((0, 1)) + 1e-8))
        return np.concatenate(parts, -1)

    x = {s: design(fold.idx[s]) for s in ("train", "val", "test")}
    xt = x["train"].reshape(-1, x["train"].shape[-1])
    preds = {s: np.zeros((len(fold.idx[s]), N, HORIZON)) for s in ("val", "test")}
    for k in range(HORIZON):
        yt = (lc[fold.idx["train"] + k] - lc[fold.idx["train"] - 1]).reshape(-1)
        m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                          min_samples_leaf=40, random_state=0).fit(xt, yt)
        smear = float(np.mean(np.exp(yt - m.predict(xt))))
        for s in preds:
            g = m.predict(x[s].reshape(-1, x[s].shape[-1])).reshape(len(fold.idx[s]), N)
            preds[s][..., k] = np.clip((np.nan_to_num(CASES)[fold.idx[s] - 1] + 1.0) * np.exp(g) * smear - 1.0, 0, None)
    truth = {s: np.stack([CASES[i: i + HORIZON].T for i in fold.idx[s]]) for s in preds}
    row = score(preds["test"], truth["test"])
    row.update(val_RMSE=rmse(preds["val"], truth["val"]))
    persist = {s: np.repeat(CASES[fold.idx[s] - 1][:, :, None], HORIZON, 2) for s in preds}
    return row, {"pred": preds, "truth": truth, "persist": persist}

### 6.2 Synthetic training data

* **TimeGAN** (Yoon et al. 2019), fitted to the analogue-safe training windows of each
  fold: embedder/recovery, supervisor, then joint adversarial training with the
  paper's loss weights. Its samples join the training set only.
* **SEIR-simulated epidemics**: real starting states, a random-walk force of
  infection whose centre and step are calibrated to training-window growth, and
  negative-binomial reporting noise. The model pre-trains on them for 50 epochs.

In [ ]:
def to_pack(x_raw, y_raw, season, pop, src, fold):
    z = lambda a: (np.log1p(a) - fold.mean) / fold.std  # noqa: E731
    p_raw = np.repeat(x_raw[..., -1:], y_raw.shape[-1], -1)
    feats = np.concatenate([z(x_raw), np.repeat(season[:, None, :], x_raw.shape[1], 1)], -1)
    t = lambda a: torch.tensor(np.asarray(a), dtype=torch.float32)  # noqa: E731
    return {"x": t(feats), "y_raw": t(y_raw), "p_raw": t(p_raw), "y_z": t(z(y_raw)), "p_z": t(z(p_raw)),
            "x_raw": t(x_raw), "pop": t(pop), "idx": src}


def concat_packs(a, b):
    out = {k: torch.cat([a[k], b[k]]) for k in a if k != "idx"}
    out["idx"] = np.concatenate([a["idx"], b["idx"]])
    return out


def _real_sequences(fold):
    lib, w = library_idx(fold), fold.window
    c = np.nan_to_num(CASES)
    seq = np.stack([np.log1p(c[i - w: i + HORIZON]) for i in lib])
    season = np.stack([seasonal_features(np.arange(i - w, i + HORIZON)) for i in lib])
    return lib, seq, season


class _Rnn(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, layers, act=True):
        super().__init__()
        self.rnn, self.fc, self.act = nn.GRU(d_in, d_hidden, num_layers=layers, batch_first=True), \
            nn.Linear(d_hidden, d_out), act

    def forward(self, x):
        y = self.fc(self.rnn(x)[0])
        return torch.sigmoid(y) if self.act else y


def timegan(seqs, n_samples, seed, hidden=32, layers=2, iters=1500, batch=64, gamma=1.0):
    g = torch.Generator().manual_seed(seed)
    torch.manual_seed(seed)
    lo, hi = seqs.min((0, 1), keepdims=True), seqs.max((0, 1), keepdims=True)
    x_all = torch.tensor((seqs - lo) / (hi - lo + 1e-7), dtype=torch.float32)
    k, t, f = x_all.shape
    emb, rec, gen = _Rnn(f, hidden, hidden, layers), _Rnn(hidden, hidden, f, layers), _Rnn(f, hidden, hidden, layers)
    sup, dis = _Rnn(hidden, hidden, hidden, max(1, layers - 1)), _Rnn(hidden, hidden, 1, layers, act=False)
    bce, mse = nn.BCEWithLogitsLoss(), nn.MSELoss()
    batch_x = lambda: x_all[torch.randint(0, k, (min(batch, k),), generator=g)]  # noqa: E731
    noise = lambda m: torch.rand(m, t, f, generator=g)  # noqa: E731
    opt_er = torch.optim.Adam([*emb.parameters(), *rec.parameters()], lr=1e-3)
    for _ in range(iters):
        x = batch_x()
        loss = 10 * torch.sqrt(mse(rec(emb(x)), x))
        opt_er.zero_grad(); loss.backward(); opt_er.step()
    opt_s = torch.optim.Adam(sup.parameters(), lr=1e-3)
    for _ in range(iters):
        h = emb(batch_x()).detach()
        loss = mse(sup(h)[:, :-1], h[:, 1:])
        opt_s.zero_grad(); loss.backward(); opt_s.step()
    opt_g = torch.optim.Adam([*gen.parameters(), *sup.parameters()], lr=1e-3)
    opt_e = torch.optim.Adam([*emb.parameters(), *rec.parameters()], lr=1e-3)
    opt_d = torch.optim.Adam(dis.parameters(), lr=1e-3)
    for _ in range(iters):
        for _ in range(2):
            x = batch_x(); zz = noise(len(x))
            h, e_hat = emb(x), gen(zz)
            h_hat = sup(e_hat); x_hat = rec(h_hat)
            y_fake, y_fake_e = dis(h_hat), dis(e_hat)
            g_u = bce(y_fake, torch.ones_like(y_fake)) + gamma * bce(y_fake_e, torch.ones_like(y_fake_e))
            g_s = mse(sup(h)[:, :-1], h[:, 1:])
            g_v = (x_hat.std(0) - x.std(0)).abs().mean() + (x_hat.mean(0) - x.mean(0)).abs().mean()
            loss = g_u + 100 * torch.sqrt(g_s) + 100 * g_v
            opt_g.zero_grad(); loss.backward(); opt_g.step()
            x = batch_x(); h = emb(x)
            e_loss = 10 * torch.sqrt(mse(rec(h), x)) + 0.1 * mse(sup(h)[:, :-1], h[:, 1:])
            opt_e.zero_grad(); e_loss.backward(); opt_e.step()
        x = batch_x(); zz = noise(len(x))
        h, e_hat = emb(x).detach(), gen(zz).detach()
        h_hat = sup(e_hat).detach()
        y_real, y_fake, y_fake_e = dis(h), dis(h_hat), dis(e_hat)
        d_loss = (bce(y_real, torch.ones_like(y_real)) + bce(y_fake, torch.zeros_like(y_fake))
                  + gamma * bce(y_fake_e, torch.zeros_like(y_fake_e)))
        if d_loss.item() > 0.15:
            opt_d.zero_grad(); d_loss.backward(); opt_d.step()
    with torch.no_grad():
        out = rec(sup(gen(noise(n_samples)))).numpy()
    return out * (hi - lo + 1e-7) + lo


def synth_timegan(fold, n, seed, iters=1500):
    lib, seq, season = _real_sequences(fold)
    w = fold.window
    fake = timegan(np.concatenate([seq, season], -1), n, seed, iters=iters)
    counts = np.expm1(np.clip(fake[..., :N], 0.0, None))
    src = np.random.default_rng(seed).choice(lib, n)
    return to_pack(counts[:, :w].transpose(0, 2, 1), counts[:, w:].transpose(0, 2, 1),
                   np.clip(fake[:, w, N:], -1.0, 1.0), POPULATION[src - 1], src, fold)


def _lambda_stats(fold):
    lib = library_idx(fold)
    c = np.nan_to_num(CASES)
    x_raw = torch.tensor(np.stack([c[i - fold.window: i].T for i in lib]), dtype=torch.float32)
    y = torch.tensor(np.stack([c[i: i + HORIZON].T for i in lib]), dtype=torch.float32)
    pop = torch.tensor(POPULATION[lib - 1], dtype=torch.float32)
    st = seir_state(x_raw, pop, cumulative(lib))
    lams = []
    for h in range(HORIZON):
        lam = invert(st, pop, y[..., h])
        lams.append(lam)
        st = advance(st, lam)
    lam = torch.stack(lams, -1).numpy()
    log_lam = np.log(np.where(lam > 1e-8, lam, np.nan))
    return float(np.nanmedian(log_lam)), float(np.nanstd(np.diff(log_lam, axis=-1)))


def _simulate(fold, n, seed, alpha, med, step):
    rng = np.random.default_rng(seed)
    lib, w = library_idx(fold), fold.window
    c = np.nan_to_num(CASES)
    src = rng.choice(lib, n)
    x0 = torch.tensor(np.stack([c[i - w: i].T for i in src]), dtype=torch.float32)
    pop = POPULATION[src - 1]
    state0 = seir_state(x0, torch.tensor(pop, dtype=torch.float32), cumulative(src))
    walk = (np.cumsum(rng.normal(0, step, (n, 1, w + HORIZON)), -1)
            + np.cumsum(rng.normal(0, step, (n, N, w + HORIZON)), -1))
    lam = torch.tensor(np.exp(np.clip(med + walk, -25.0, LOG_LAMBDA_MAX)), dtype=torch.float32)
    _, inc = simulate_weeks(state0, lam)
    mu = (inc.numpy() * RHO * pop[..., None]).clip(1e-9, None)
    counts = rng.poisson(rng.gamma(shape=1.0 / alpha, scale=mu * alpha)).astype(np.float64)
    return counts[..., :w], counts[..., w:], seasonal_features(src + w), pop, src


def synth_seir(fold, n, seed, alpha=0.2):
    """Calibrated: the log-lambda centre and step are chosen so simulated growth matches training growth."""
    _, seq, _ = _real_sequences(fold)
    g = np.diff(np.log1p(np.expm1(seq).transpose(0, 2, 1)), axis=-1)
    target = (g.mean(), g.std())
    med, sd = _lambda_stats(fold)
    best = None
    for off in np.arange(-4.0, 1.01, 0.25):
        for mult in (0.05, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0):
            xs, ys, *_ = _simulate(fold, 400, 12345, 0.2, med + off, sd / np.sqrt(2.0) * mult)
            gs = np.diff(np.log1p(np.concatenate([xs, ys], -1)), axis=-1)
            err = (gs.mean() - target[0]) ** 2 + (gs.std() - target[1]) ** 2
            if best is None or err < best[0]:
                best = (err, off, mult)
    x_raw, y_raw, season, pop, src = _simulate(fold, n, seed, alpha, med + best[1], sd / np.sqrt(2.0) * best[2])
    return to_pack(x_raw, y_raw, season, pop, src, fold)

## 7. Every configuration, trained here

Each configuration the paper reports is defined once below and trained on every
(origin, seed). Because everything runs in one environment with one code path, any
two configurations evaluated on the same origins can be paired directly — a shared
control (the best model, **B**) is trained once and used by every comparison that
needs it.

* **Three origins** (0.55, 0.70, 0.85) × seeds 0, 1, 2 — the screening protocol.
* **Nine disjoint origins** (0.40 + k/15) × seeds 0, 1, 2 — the confirmation, where
  the origin-level paired test can reach p = 0.004.

In [ ]:
ENC = ["AAGCN", "ASTGCN", "LSTM", "STGAT", "A3TGCN", "DCRNN"]
PHYS = dict(lam_param="log", state_fit=True)
NBS = dict(loss="nb", dist="nb", use_season=True, epochs=400)            # NB likelihood + seasonal features
B = dict(backbone="AAGCN", head="direct", **NBS)                          # the best model on validation
BLOCKS_3 = ((2, 5), (6, 9), (10, 13))
BLOCKS_6 = ((2, 5), (6, 9), (10, 13), (14, 17), (18, 21), (22, 25))

CONFIGS3: dict[str, dict] = {}
# Six encoders x five heads, squared error, no seasonal features (paper Tables 2 and 5).
for e in ENC:
    for h in ("direct", "residual", "gated", "foi", "foi_res"):
        CONFIGS3[f"{e}+{h}"] = dict(backbone=e, head=h, loss="mse_z", epochs=300, **PHYS)
# The first screen, on a toy graph layer: does the graph help, does the season help?
for g in ("gcn", "none", "adaptive"):
    CONFIGS3[f"graph={g}"] = dict(backbone=g, head="residual", loss="mse_z", epochs=300)
CONFIGS3["feat=season"] = dict(backbone="gcn", head="residual", loss="mse_z", use_season=True, epochs=300)
# The best model and the likelihood comparison.
CONFIGS3["B"] = B
CONFIGS3["B, squared error"] = dict(B, loss="mse_z", dist="point")
for e in ("AAGCN", "ASTGCN", "LSTM"):
    CONFIGS3[f"{e}+foi_res, NB+season"] = dict(backbone=e, head="foi_res", **NBS, **PHYS)
for e in ("ASTGCN", "LSTM", "STGAT", "A3TGCN"):
    CONFIGS3[f"{e}+direct, NB+season"] = dict(backbone=e, head="direct", **NBS)
# Architecture changes to B.
CONFIGS3["A1 residual head"] = dict(B, head="residual")
CONFIGS3["A2 district seasonal curves"] = dict(B, dseason=True)
CONFIGS3["A3 MLP head"] = dict(B, head_mlp=64)
CONFIGS3["A4 MLP head + climate 2-13"] = dict(B, head_mlp=64, clim_blocks=BLOCKS_3)
CONFIGS3["A5 global context"] = dict(B, global_ctx=True)
CONFIGS3["A6 combined"] = dict(B, head_mlp=64, dseason=True, global_ctx=True, node_emb=16, clim_blocks=BLOCKS_3)
# Remedies from the forecasting literature.
CONFIGS3["R1 RevIN"] = dict(B, norm="revin")
CONFIGS3["R2 district embedding"] = dict(B, node_emb=16)
CONFIGS3["R3 STID-style MLP"] = dict(B, backbone="none", node_emb=16)
CONFIGS3["R4a NB-GLM"] = dict(B, backbone="linear", node_emb=8)
CONFIGS3["R4b k-NN"] = dict(kind="knn")
CONFIGS3["R6 SEIR auxiliary loss"] = dict(B, aux_phys=0.1, **PHYS)
# Data.
for frac in (0.25, 0.5, 0.75):
    CONFIGS3[f"B, {int(frac * 100)}% of training"] = dict(B, train_frac=frac)
CONFIGS3["G2 TimeGAN"] = dict(B, augment="timegan")
CONFIGS3["G3 SEIR pre-training"] = dict(B, augment="seir_pretrain_cal")
CONFIGS3["K1 climate lags 2-4"] = dict(B, use_climate=True)
CONFIGS3["K2 climate lags 2-13"] = dict(B, clim_blocks=BLOCKS_3)
CONFIGS3["K3 climate lags 2-25"] = dict(B, clim_blocks=BLOCKS_6)
CONFIGS3["K6 trees + climate"] = dict(kind="gbm", clim_blocks=BLOCKS_3)
CONFIGS3["K7 trees"] = dict(kind="gbm")
# Physics as structure.
CONFIGS3["P1 spatial penalty"] = dict(B, spatial=0.001, spatial_kind="ratio")
CONFIGS3["P2 spatial penalty (log)"] = dict(B, spatial=0.05, spatial_kind="log")
CONFIGS3["P4 metapopulation SEIR"] = dict(backbone="AAGCN", head="foi_meta", **NBS, **PHYS)
CONFIGS3["P5 metapopulation + penalty + season"] = dict(backbone="AAGCN", head="foi_meta", spatial=0.001,
                                                        dseason=True, **NBS, **PHYS)

CONFIGS9 = {
    "B": B,
    "LSTM+direct, NB+season": dict(backbone="LSTM", head="direct", **NBS),
    "ASTGCN+direct, NB+season": dict(backbone="ASTGCN", head="direct", **NBS),
    "AAGCN+foi_res, NB+season": dict(backbone="AAGCN", head="foi_res", **NBS, **PHYS),
    "ASTGCN+foi_res, NB+season": dict(backbone="ASTGCN", head="foi_res", **NBS, **PHYS),
    "LSTM+foi_res, NB+season": dict(backbone="LSTM", head="foi_res", **NBS, **PHYS),
    "P4 metapopulation SEIR": dict(backbone="AAGCN", head="foi_meta", **NBS, **PHYS),
}

SEEDS = (0, 1, 2)
ORIGIN_SETS = {"three": (ORIGINS, TEST_FRAC), "nine": (ORIGINS_9, TEST_FRAC_9)}
if QUICK:
    SEEDS = (0,)
    ORIGIN_SETS = {"three": ((0.70,), TEST_FRAC), "nine": ((0.40 + 4 / 15, 0.40 + 5 / 15), TEST_FRAC_9)}
FOLDS = {k: {f.origin: f for f in build_folds(o, tf)} for k, (o, tf) in ORIGIN_SETS.items()}

JOBS = [("three", name, o, s) for name in CONFIGS3 for o in FOLDS["three"] for s in SEEDS]
JOBS += [("nine", name, o, s) for name in CONFIGS9 for o in FOLDS["nine"] for s in SEEDS]
print(f"{len(CONFIGS3)} configurations x {len(FOLDS['three'])} origins x {len(SEEDS)} seeds "
      f"+ {len(CONFIGS9)} x {len(FOLDS['nine'])} x {len(SEEDS)}  =  {len(JOBS)} training runs")

In [ ]:
import pickle
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp


#: The configurations whose training-window predictions are kept for the ceiling analysis (8.6).
CEILING = {"B", "ASTGCN+direct, NB+season", "LSTM+direct, NB+season", "STGAT+direct, NB+season",
           "A3TGCN+direct, NB+season", "LSTM+foi_res, NB+season"}


def run_job(job):
    oset, name, origin, seed = job
    torch.set_num_threads(1)
    cfg = dict((CONFIGS3 if oset == "three" else CONFIGS9)[name])
    kind = cfg.pop("kind", "net")
    if QUICK:
        cfg["epochs"] = 3
    fold = FOLDS[oset][origin]
    t0 = time.time()
    if kind == "knn":
        row, extra = run_knn(fold, seed)
    elif kind == "gbm":
        row, extra = run_gbm(fold, seed, **cfg)
    else:
        row, extra = run_fold(fold, seed=seed, keep_train=oset == "three" and name in CEILING, **cfg)
    row.update(origin_set=oset, name=name, origin=origin, seed=seed, elapsed=round(time.time() - t0, 1))
    return job, row, extra


def expected_cost(job):
    """Rough relative cost, so the longest runs start first and the pool stays busy."""
    cfg = (CONFIGS3 if job[0] == "three" else CONFIGS9)[job[1]]
    return ({"DCRNN": 8, "ASTGCN": 4, "A3TGCN": 3}.get(cfg.get("backbone"), 1)
            * (6 if cfg.get("augment") in ("timegan", "seir_pretrain_cal") else 1))


LOG = OUT / "runs.jsonl"
done = {}
if LOG.exists():                         # resume an interrupted session
    for line in LOG.read_text(encoding="utf-8").splitlines():
        r = json.loads(line)
        done[(r["origin_set"], r["name"], r["origin"], r["seed"])] = r
PRED_FILE = OUT / "predictions.pkl"
PREDS = pickle.loads(PRED_FILE.read_bytes()) if PRED_FILE.exists() else {}
todo = sorted((j for j in JOBS if j not in done), key=expected_cost, reverse=True)
print(f"{len(done)} runs already on disk, {len(todo)} to train with {WORKERS} worker(s)")

t_start = time.time()
if todo:
    use_pool = WORKERS > 1 and "fork" in mp.get_all_start_methods()
    pool = ProcessPoolExecutor(WORKERS, mp_context=mp.get_context("fork")) if use_pool else None
    futures = [pool.submit(run_job, j) for j in todo] if pool else None
    stream = (f.result() for f in as_completed(futures)) if pool else (run_job(j) for j in todo)
    with LOG.open("a", encoding="utf-8") as log:
        for k, (job, row, extra) in enumerate(stream, 1):
            done[job] = row
            PREDS[job] = extra
            log.write(json.dumps(row) + "\n")
            log.flush()
            if k % 25 == 0 or k == len(todo):
                el = time.time() - t_start
                print(f"  {k}/{len(todo)} runs  {el / 60:6.1f} min elapsed, ~{el / k * (len(todo) - k) / 60:6.1f} min left",
                      flush=True)
                PRED_FILE.write_bytes(pickle.dumps(PREDS))
    if pool:
        pool.shutdown()
PRED_FILE.write_bytes(pickle.dumps(PREDS))
print(f"training finished: {len(done)} runs in {(time.time() - t_start) / 3600:.2f} h this session")

## 8. Results — computed only from the runs above

**How to read the numbers.** RMSE is in weekly cases; lower is better. Validation
decides and test is only reported: every run was early-stopped on validation, and no
choice below looks at test. Pairs are compared on matched (origin, seed) runs with an
exact two-sided sign-flip permutation test, Benjamini–Hochberg-corrected across each
table. With three origins the *origin*-level test cannot reach p < 0.25, so
three-origin p-values (9 (origin, seed) pairs) measure run-to-run stability; claims
about the series rest on the nine-origin runs.

In [ ]:
import itertools

rows = pd.DataFrame(list(done.values()))


def persistence_rows(oset):
    out = []
    for o, fold in FOLDS[oset].items():
        pk = {s: build_tensors(fold, s) for s in ("val", "test")}
        r = score(pk["test"]["p_raw"].numpy(), pk["test"]["y_raw"].numpy())
        r.update(origin_set=oset, name="persistence", origin=o, seed=-1,
                 val_RMSE=rmse(pk["val"]["p_raw"].numpy(), pk["val"]["y_raw"].numpy()))
        out.append(r)
    return out


def ensemble_rows(members, name, oset="three"):
    out = []
    for o in FOLDS[oset]:
        for s in SEEDS:
            parts = [PREDS[(oset, m, o, s)] for m in members]
            val = np.mean([p["pred"]["val"] for p in parts], 0)
            test = np.mean([p["pred"]["test"] for p in parts], 0)
            r = score(test, parts[0]["truth"]["test"])
            r.update(origin_set=oset, name=name, origin=o, seed=s, val_RMSE=rmse(val, parts[0]["truth"]["val"]))
            out.append(r)
    return out


rows = pd.concat([rows, pd.DataFrame(persistence_rows("three") + persistence_rows("nine")),
                  pd.DataFrame(ensemble_rows(["B", "R4a NB-GLM", "R4b k-NN"], "R5 ensemble B + NB-GLM + k-NN"))],
                 ignore_index=True)
R3 = rows[rows.origin_set == "three"]
R9 = rows[rows.origin_set == "nine"]
rows.to_json(OUT / "runs.json", orient="records", indent=1)


def diffs(df, arm, ref, metric="val_RMSE", unit="origin_seed"):
    a = df[df.name == arm].groupby(["origin", "seed"])[metric].mean()
    r = df[df.name == ref].groupby(["origin", "seed"])[metric].mean()
    if (r.index.get_level_values("seed") < 0).all():             # persistence: one row per origin
        r = pd.Series({(o, s): r.xs(o, level="origin").iloc[0] for o, s in a.index})
    d = (a - r).dropna()
    return d.groupby(level="origin").mean().to_numpy() if unit == "origin" else d.to_numpy()


def sign_flip_p(d, n_perm=100_000, seed=42):
    d = np.asarray(d, float)
    flips = (np.array(list(itertools.product((-1, 1), repeat=len(d))), float) if len(d) <= 16
             else np.random.default_rng(seed).choice((-1.0, 1.0), size=(n_perm, len(d))))
    return float((np.abs(flips @ d / len(d)) >= abs(d.mean()) - 1e-12).mean())


def bh(p):
    p = np.asarray(p, float); m = len(p); adj = np.empty(m); run = 1.0
    for rank, i in reversed(list(enumerate(np.argsort(p), start=1))):
        run = min(run, p[i] * m / rank); adj[i] = min(run, 1.0)
    return adj


def compare(df, arm, ref, metric="val_RMSE", unit="origin_seed"):
    d = diffs(df, arm, ref, metric, unit)
    return {"delta": float(d.mean()), "wins": f"{int((d < 0).sum())}/{len(d)}", "p": sign_flip_p(d)}


def mean_of(df, name, metric="val_RMSE"):
    return float(df.loc[df.name == name, metric].mean())


pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
RESULTS = {}           # every headline number, written to outputs/results.json at the end

### 8.1 Persistence, the floor every model competes with

The forecast "next three weeks = last observed week". Last week's cases explain almost
all of this week's; the best climate variable at a causal lag explains almost nothing.

In [ ]:
def r2_pooled(x, y):
    m = ~(np.isnan(x) | np.isnan(y))
    return float(np.corrcoef(x[m], y[m])[0, 1] ** 2)


RESULTS["r2_lag1"] = r2_pooled(CASES[:-1].ravel(), CASES[1:].ravel())
RESULTS["r2_best_climate"] = max(r2_pooled(CLIMATE[:-lag, :, c].ravel(), CASES[lag:].ravel())
                                 for c in range(CLIMATE.shape[-1]) for lag in range(2, 9))
RESULTS["persistence_3"] = {"val": mean_of(R3, "persistence"), "test": mean_of(R3, "persistence", "RMSE")}
RESULTS["persistence_9"] = {"val": mean_of(R9, "persistence"), "test": mean_of(R9, "persistence", "RMSE")}
RESULTS["array_floor"] = {"all_windows": float(ARRAY_FLOOR[0]), "without_row_395": float(ARRAY_FLOOR[1])}
print(f"cases(t-1) -> cases(t): r2 = {RESULTS['r2_lag1']:.3f};  best climate variable (lags 2-8): "
      f"r2 = {RESULTS['r2_best_climate']:.3f}")
print(f"persistence, three origins: val {RESULTS['persistence_3']['val']:.2f}  test {RESULTS['persistence_3']['test']:.2f}")
print(f"persistence, nine origins:  val {RESULTS['persistence_9']['val']:.2f}  test {RESULTS['persistence_9']['test']:.2f}")

### 8.2 Six encoders, one harness (paper Table 2)

In [ ]:
tab = pd.DataFrame({h: [f"{mean_of(R3, f'{e}+{h}'):.2f} / {mean_of(R3, f'{e}+{h}', 'RMSE'):.2f}" for e in ENC]
                    for h in ("direct", "foi", "foi_res")}, index=ENC)
tab.columns = ["direct", "SEIR decoder", "gated SEIR"]
print("mean validation / test RMSE; persistence "
      f"{RESULTS['persistence_3']['val']:.2f} / {RESULTS['persistence_3']['test']:.2f}")
print(tab.to_string())
enc_pairs = []
for e in ENC:
    for h in ("foi", "foi_res"):
        v, t = compare(R3, f"{e}+{h}", f"{e}+direct"), compare(R3, f"{e}+{h}", f"{e}+direct", "RMSE")
        enc_pairs.append({"encoder": e, "head": h, "val delta": v["delta"], "val wins": v["wins"], "p": v["p"],
                          "test delta": t["delta"], "test wins": t["wins"]})
ENC_PAIRS = pd.DataFrame(enc_pairs)
print("\neach SEIR head against the same encoder's direct head:")
print(ENC_PAIRS.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
x = np.arange(len(ENC))
for ax, metric, title in ((axes[0], "val_RMSE", "validation"), (axes[1], "RMSE", "test")):
    for k, (h, lab) in enumerate((("direct", "direct"), ("foi", "SEIR decoder"), ("foi_res", "gated SEIR"))):
        ax.bar(x + (k - 1) * 0.27, [mean_of(R3, f"{e}+{h}", metric) for e in ENC], 0.27, label=lab)
    ax.axhline(mean_of(R3, "persistence", metric), color="k", ls="--", lw=1, label="persistence")
    ax.set_xticks(x, ENC); ax.set_title(title); ax.set_ylabel("RMSE")
axes[0].legend(fontsize=8, ncol=2); fig.tight_layout(); fig.savefig(OUT / "fig_encoders.png", dpi=150); plt.show()

### 8.3 Anchor, gate or physics? (paper Table 5)

The gated SEIR head differs from the direct head in three ways: a persistence anchor, a
learned gate, and the SEIR simulator. The pre-registered rule credits a repair to the
physics only if gated SEIR beats its no-physics twin (`gated`) with ≥ 7/9 wins and
BH-adjusted p < 0.05 on each failing encoder.

In [ ]:
H4 = ["direct", "residual", "gated", "foi_res"]
print(pd.DataFrame({h: [mean_of(R3, f"{e}+{h}") for e in ENC] for h in H4}, index=ENC).to_string())
res = []
for e in ENC:
    v, t = compare(R3, f"{e}+foi_res", f"{e}+gated"), compare(R3, f"{e}+foi_res", f"{e}+gated", "RMSE")
    d, g, f = (mean_of(R3, f"{e}+{h}") for h in ("direct", "gated", "foi_res"))
    res.append({"encoder": e, "SEIR - gated (val)": v["delta"], "wins": v["wins"], "p": v["p"],
                "SEIR - gated (test)": t["delta"], "test wins": t["wins"],
                "share of repair from anchor+gate": (d - g) / (d - f) if abs(d - f) > 1e-9 else np.nan})
RESCUE = pd.DataFrame(res)
failing = RESCUE.encoder.isin(["STGAT", "A3TGCN", "DCRNN"])
RESCUE.loc[failing, "p_adj"] = bh(RESCUE.loc[failing, "p"])
print("\n" + RESCUE.to_string(index=False))
credited = [r["encoder"] for _, r in RESCUE[failing].iterrows()
            if r["SEIR - gated (val)"] < 0 and int(r["wins"].split("/")[0]) >= 7 and r["p_adj"] < 0.05]
print(f"\nencoders whose repair is credited to the SEIR simulator: {credited or 'none'}")

### 8.4 Every lever, paired against its control (paper Table 3)

In [ ]:
LEVERS = [
    ("worked", "NB likelihood (vs squared error)", "B", "B, squared error"),
    ("worked", "seasonal features", "feat=season", "graph=gcn"),
    ("worked", "best model B vs persistence", "B", "persistence"),
    ("graph/architecture", "graph (GCN vs no message passing)", "graph=gcn", "graph=none"),
    ("graph/architecture", "learned adjacency", "graph=adaptive", "graph=gcn"),
    ("graph/architecture", "district seasonal curves", "A2 district seasonal curves", "B"),
    ("graph/architecture", "nonlinear (MLP) head", "A3 MLP head", "B"),
    ("graph/architecture", "MLP head + climate lags 2-13", "A4 MLP head + climate 2-13", "B"),
    ("graph/architecture", "global context node", "A5 global context", "B"),
    ("graph/architecture", "residual head + NB", "A1 residual head", "B"),
    ("graph/architecture", "combined architecture changes", "A6 combined", "B"),
    ("literature remedy", "RevIN", "R1 RevIN", "B"),
    ("literature remedy", "district identity embedding", "R2 district embedding", "B"),
    ("literature remedy", "STID-style MLP", "R3 STID-style MLP", "B"),
    ("literature remedy", "NB-GLM (no network)", "R4a NB-GLM", "B"),
    ("literature remedy", "k-NN analogues (no network)", "R4b k-NN", "B"),
    ("literature remedy", "ensemble B + NB-GLM + k-NN", "R5 ensemble B + NB-GLM + k-NN", "B"),
    ("data", "half the training windows", "B, 50% of training", "B"),
    ("data", "TimeGAN augmentation", "G2 TimeGAN", "B"),
    ("data", "SEIR-simulated pre-training", "G3 SEIR pre-training", "B"),
    ("data", "climate, lags 2-4", "K1 climate lags 2-4", "B"),
    ("data", "climate, lags 2-13", "K2 climate lags 2-13", "B"),
    ("data", "climate, lags 2-25", "K3 climate lags 2-25", "B"),
    ("physics", "SEIR as auxiliary loss", "R6 SEIR auxiliary loss", "B"),
    ("physics", "SEIR simulator vs no-physics twin (AAGCN)", "AAGCN+foi_res", "AAGCN+gated"),
    ("physics", "spatial physics penalty", "P1 spatial penalty", "B"),
    ("physics", "metapopulation vs gated SEIR head", "P4 metapopulation SEIR", "AAGCN+foi_res, NB+season"),
    ("physics", "gated SEIR-GNN vs SEIR-LSTM", "AAGCN+foi_res, NB+season", "LSTM+foi_res, NB+season"),
]
lev = []
for group, label, arm, ref in LEVERS:
    v, t = compare(R3, arm, ref), compare(R3, arm, ref, "RMSE")
    lev.append({"group": group, "lever": label, "arm": arm, "control": ref, "val delta": v["delta"],
                "wins": v["wins"], "p": v["p"], "test delta": t["delta"]})
LEV = pd.DataFrame(lev)
LEV["p_adj"] = bh(LEV.p)
LEV["consistent gain"] = (LEV["val delta"] < 0) & (LEV.wins.str.split("/").str[0].astype(int) >= 7)
print(LEV.drop(columns=["arm", "control"]).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 7))
y = np.arange(len(LEV))[::-1]
ax.barh(y, LEV["val delta"], color=np.where(LEV["consistent gain"], "#2a9d8f",
                                             np.where(LEV["val delta"] < 0, "#9aa5b1", "#e76f51")))
ax.axvline(0, color="k", lw=0.8); ax.set_yticks(y, LEV.lever, fontsize=8)
ax.set_xlabel("validation RMSE change vs own control (negative = better)")
fig.tight_layout(); fig.savefig(OUT / "fig_levers.png", dpi=150); plt.show()

extra = {"trees gain from climate": compare(R3, "K6 trees + climate", "K7 trees"),
         "learning curve 25% vs 100%": compare(R3, "B, 25% of training", "B"),
         "learning curve 75% vs 100%": compare(R3, "B, 75% of training", "B")}
for k, v in extra.items():
    print(f"{k}: {v['delta']:+.3f} ({v['wins']}, p {v['p']:.3f})")

### 8.5 The best model against persistence, and SEIR-GNN against SEIR-LSTM (paper Table 4)

In [ ]:
cmp = []
for label, df, arm, ref, unit in [
    ("B vs persistence, 3 origins", R3, "B", "persistence", "origin_seed"),
    ("B vs persistence, 9 origins", R9, "B", "persistence", "origin"),
    ("gated SEIR-GNN (ASTGCN) vs SEIR-LSTM, 3 origins", R3, "ASTGCN+foi_res, NB+season", "LSTM+foi_res, NB+season", "origin_seed"),
    ("gated SEIR-GNN (AAGCN) vs SEIR-LSTM, 3 origins", R3, "AAGCN+foi_res, NB+season", "LSTM+foi_res, NB+season", "origin_seed"),
    ("AAGCN vs LSTM, direct head, 3 origins", R3, "B", "LSTM+direct, NB+season", "origin_seed"),
    ("gated SEIR-GNN (ASTGCN) vs SEIR-LSTM, 9 origins", R9, "ASTGCN+foi_res, NB+season", "LSTM+foi_res, NB+season", "origin"),
    ("gated SEIR-GNN (AAGCN) vs SEIR-LSTM, 9 origins", R9, "AAGCN+foi_res, NB+season", "LSTM+foi_res, NB+season", "origin"),
    ("metapopulation SEIR-GNN vs SEIR-LSTM, 9 origins", R9, "P4 metapopulation SEIR", "LSTM+foi_res, NB+season", "origin"),
    ("AAGCN vs LSTM, direct head, 9 origins", R9, "B", "LSTM+direct, NB+season", "origin"),
    ("gated SEIR-GNN (AAGCN) vs persistence, 9 origins", R9, "AAGCN+foi_res, NB+season", "persistence", "origin"),
]:
    v, t = compare(df, arm, ref, "val_RMSE", unit), compare(df, arm, ref, "RMSE", unit)
    cmp.append({"comparison": label, "unit": unit, "val delta": v["delta"], "val wins": v["wins"], "val p": v["p"],
                "test delta": t["delta"], "test wins": t["wins"], "test p": t["p"]})
CMP = pd.DataFrame(cmp)
print(CMP.to_string(index=False))

origins9 = sorted(R9.origin.unique())
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, metric, title in ((axes[0], "val_RMSE", "validation"), (axes[1], "RMSE", "test")):
    for arm in ("B", "AAGCN+foi_res, NB+season", "LSTM+foi_res, NB+season", "persistence"):
        ax.plot(origins9, [R9[(R9.name == arm) & (R9.origin == o)][metric].mean() for o in origins9],
                marker="o", ms=3, ls="--" if arm == "persistence" else "-", label=arm)
    ax.set_yscale("log"); ax.set_title(f"nine origins, {title}"); ax.set_xlabel("origin")
axes[1].legend(fontsize=8); fig.tight_layout(); fig.savefig(OUT / "fig_nine.png", dpi=150); plt.show()

### 8.6 The ceiling: every family makes the same errors

Residuals (forecast − truth) on the validation windows of the best configuration of
each encoder (negative-binomial likelihood, seasonal features, direct head) and of the
SEIR-LSTM, correlated cell by cell; and how much of each model's validation residual a
linear model on the same inputs can still explain.

In [ ]:
members = {"AAGCN": "B", "ASTGCN": "ASTGCN+direct, NB+season", "LSTM": "LSTM+direct, NB+season",
           "STGAT": "STGAT+direct, NB+season", "A3TGCN": "A3TGCN+direct, NB+season",
           "SEIR-LSTM": "LSTM+foi_res, NB+season", "k-NN": "R4b k-NN"}
resid = {}
for lab, name in members.items():
    resid[lab] = np.concatenate([(PREDS[("three", name, o, s)]["pred"]["val"]
                                  - PREDS[("three", name, o, s)]["truth"]["val"]).ravel()
                                 for o in FOLDS["three"] for s in SEEDS])
RESID_CORR = pd.DataFrame(np.corrcoef(np.stack(list(resid.values()))), index=list(resid), columns=list(resid))
print(RESID_CORR.round(3).to_string())
RESULTS["residual_corr_range_vs_B"] = [float(RESID_CORR["AAGCN"].drop("AAGCN").min()),
                                       float(RESID_CORR["AAGCN"].drop("AAGCN").max())]


def pooled(name, split):
    parts = [PREDS[("three", name, o, s)] for o in FOLDS["three"] for s in SEEDS]
    cat = lambda k: np.concatenate([p[k][split] for p in parts])  # noqa: E731
    return {"pred": cat("pred"), "truth": cat("truth"), "persist": cat("persist"), "idx": cat("idx")}


# How the errors look: a lag on moves, and concentration in a few district-weeks.
# A week "moves" when log growth from the last observed week exceeds 0.3 (about 35%).
ceil = []
for lab, name in list(members.items()) + [("persistence", None)]:
    v = pooled(members["AAGCN"] if name is None else name, "val")
    pred = v["persist"] if name is None else v["pred"]
    g = np.log1p(v["truth"]) - np.log1p(v["persist"])
    err, sse = pred - v["truth"], (pred - v["truth"]) ** 2
    flat = np.sort(sse.ravel())[::-1]
    ceil.append({"model": lab, "bias on rising weeks": err[g > 0.3].mean(), "bias on falling weeks": err[g < -0.3].mean(),
                 "share of squared error in the top 5% of cells": flat[: len(flat) // 20].sum() / flat.sum()})
CEIL = pd.DataFrame(ceil).set_index("model")
print("\n" + CEIL.to_string())


# Signal left on the table: fit a linear model to each model's TRAINING residuals (log space) on
# what the model was given plus its own forecast, and score it on VALIDATION.
def residual_features(part):
    c = np.log1p(np.nan_to_num(CASES))
    hist = np.stack([c[i - WINDOW: i].T for i in part["idx"]])
    nbr = np.einsum("ij,kjw->kiw", FIXED.numpy(), hist)[..., -1:]
    season = np.repeat(seasonal_features(part["idx"])[:, None, :], N, 1)
    pred = np.log1p(part["pred"])
    out = []
    for h in range(HORIZON):
        onehot = np.zeros(pred.shape[:2] + (HORIZON,))
        onehot[..., h] = 1
        out.append(np.concatenate([hist, nbr, season, pred[..., h:h + 1], onehot], -1))
    return np.stack(out, 2).reshape(-1, out[0].shape[-1])


recover = {}
for lab, name in members.items():
    if name not in CEILING:
        continue
    tr, va = pooled(name, "train"), pooled(name, "val")
    xt, xv = residual_features(tr), residual_features(va)
    yt = (np.log1p(tr["truth"]) - np.log1p(tr["pred"])).ravel()
    yv = (np.log1p(va["truth"]) - np.log1p(va["pred"])).ravel()
    beta = np.linalg.lstsq(np.c_[xt, np.ones(len(xt))], yt, rcond=None)[0]
    res_v = yv - np.c_[xv, np.ones(len(xv))] @ beta
    recover[lab] = float(1 - np.sum(res_v ** 2) / np.sum((yv - yv.mean()) ** 2))
RECOVER = pd.Series(recover, name="validation residual variance a linear model recovers")
print("\n" + RECOVER.round(3).to_string())

# TimeGAN's samples against the real training windows.
tg = R3[R3.name == "G2 TimeGAN"]
RESULTS["timegan_max"] = {"synthetic": float(tg.synthetic_max.max()), "real": float(tg.real_train_max.max())}
print(f"\nlargest weekly count: TimeGAN samples {RESULTS['timegan_max']['synthetic']:,.0f}, "
      f"real training windows {RESULTS['timegan_max']['real']:,.0f}")

### 8.7 Physics as structure (paper section 5.5)

In [ ]:
for arm, ref in (("P1 spatial penalty", "B"), ("P2 spatial penalty (log)", "B"),
                 ("P4 metapopulation SEIR", "AAGCN+foi_res, NB+season"),
                 ("P5 metapopulation + penalty + season", "AAGCN+foi_res, NB+season")):
    r = compare(R3, arm, ref)
    print(f"{arm:40s} vs {ref:28s} {r['delta']:+.3f}  {r['wins']}  p {r['p']:.4f}")

# Moran's I of log growth on the district graph: is there spatial structure for a
# coupling to learn? (log growth = log R over one week, model-free.)
lg = np.diff(np.log1p(CASES), axis=0)
w = ((FIXED > 0).numpy() & ~np.eye(N, dtype=bool)).astype(float)


def morans_i(v):
    ok = np.isfinite(v)
    if ok.sum() < N:
        return np.nan
    z = v - v.mean()
    return float(N / w.sum() * (z @ w @ z) / (z @ z)) if (z @ z) > 0 else np.nan


RESULTS["morans_i_log_growth"] = float(np.nanmean([morans_i(r) for r in lg]))
RESULTS["morans_i_log_cases"] = float(np.nanmean([morans_i(np.log1p(r)) for r in CASES]))
print(f"Moran's I on the district graph, mean over weeks: log cases {RESULTS['morans_i_log_cases']:.3f}, "
      f"weekly log growth {RESULTS['morans_i_log_growth']:.3f}")

### 8.8 Validation and test disagree

For each family of runs: does the configuration validation ranks first also do well
on test?

In [ ]:
families = {
    "encoders x heads": [f"{e}+{h}" for e in ENC for h in ("direct", "residual", "gated", "foi", "foi_res")],
    "screen": ["graph=gcn", "graph=none", "graph=adaptive", "feat=season"],
    "likelihood and heads": ["B", "B, squared error", "AAGCN+foi_res, NB+season", "ASTGCN+foi_res, NB+season",
                             "LSTM+foi_res, NB+season", "ASTGCN+direct, NB+season", "LSTM+direct, NB+season"],
    "architecture": ["B"] + [n for n in CONFIGS3 if re.match(r"A\d ", n)],
    "remedies": ["B"] + [n for n in rows.name.unique() if re.match(r"R\d", str(n))],
    "data": ["B"] + [n for n in CONFIGS3 if re.match(r"(G\d|K\d|B, )", n)],
    "physics": ["B"] + [n for n in CONFIGS3 if re.match(r"P\d", n)] + ["AAGCN+foi_res, NB+season",
                                                                     "LSTM+foi_res, NB+season"],
    "nine origins": [n for n in CONFIGS9],
}
vt = []
for fam, names in families.items():
    df = R9 if fam == "nine origins" else R3
    m = df[df.name.isin(names)].groupby("name")[["val_RMSE", "RMSE"]].mean()
    pick, best_t = m.val_RMSE.idxmin(), m.RMSE.idxmin()
    vt.append({"family": fam, "configs": len(m), "rank corr (val, test)": m.val_RMSE.rank().corr(m.RMSE.rank()),
               "picked by validation": pick, "its test": m.loc[pick, "RMSE"], "best on test": best_t,
               "best test": m.loc[best_t, "RMSE"], "persistence test": mean_of(df, "persistence", "RMSE")})
VT = pd.DataFrame(vt)
print(VT.to_string(index=False))
anchored = re.compile(r"foi_res|residual|gated|k-NN|foi_meta|metapopulation|SEIR")
print(f"\nfamilies where validation's pick is worse than persistence on test: "
      f"{int((VT['its test'] > VT['persistence test']).sum())} of {len(VT)}")
print(f"families whose best-on-test configuration is anchored to last week: "
      f"{sum(bool(anchored.search(n)) for n in VT['best on test'])} of {len(VT)}")

## 9. Summary and outputs

Everything below is written to `outputs/`: every run (`runs.json`, one row per
configuration × origin × seed), the predictions (`predictions.pkl`), the tables as
CSV, the figures, and `results.json` with the headline numbers. The paper's tables
and figures are regenerated from these files.

In [ ]:
LEV.to_csv(OUT / "levers.csv", index=False)
RESCUE.to_csv(OUT / "rescue.csv", index=False)
CMP.to_csv(OUT / "comparisons.csv", index=False)
ENC_PAIRS.to_csv(OUT / "encoder_heads.csv", index=False)
VT.to_csv(OUT / "validation_vs_test.csv", index=False)
DIAG.to_csv(OUT / "seir_decoder_diagnosis.csv")
RESID_CORR.to_csv(OUT / "residual_correlation.csv")
CEIL.to_csv(OUT / "error_structure.csv")
RECOVER.to_csv(OUT / "residual_recovery.csv")
RESULTS.update(profile=PROFILE, runs=int(len(done)), hours=round((time.time() - NOTEBOOK_START) / 3600, 2),
               credited_to_physics=credited)
(OUT / "results.json").write_text(json.dumps(RESULTS, indent=1, default=float), encoding="utf-8")

b, p = RESULTS["persistence_3"], RESULTS["persistence_9"]
lev = LEV.set_index("lever")
cmp_ = CMP.set_index("comparison")
print("SUMMARY" + (" -- QUICK PROFILE, NOT RESULTS" if QUICK else ""))
print(f"  data: {T} weeks rebuilt from {len(SOURCES)} verified source files; lag-1 r2 {RESULTS['r2_lag1']:.2f}, "
      f"best climate r2 {RESULTS['r2_best_climate']:.2f}")
print(f"  benchmark array: persistence {ARRAY_FLOOR[0]:.2f} (all windows) / {ARRAY_FLOOR[1]:.2f} (without row 395); "
      f"2023 rows inside every training split")
print(f"  persistence (3 origins): val {b['val']:.2f}, test {b['test']:.2f}")
print(f"  best model B: val {mean_of(R3, 'B'):.2f}, test {mean_of(R3, 'B', 'RMSE'):.2f}")
print(f"  NB likelihood: {lev.loc['NB likelihood (vs squared error)', 'val delta']:+.2f} "
      f"({lev.loc['NB likelihood (vs squared error)', 'wins']})")
print(f"  SEIR repair credited to physics: {credited or 'none'}")
print(f"  gated SEIR-GNN vs SEIR-LSTM, 9 origins: val "
      f"{cmp_.loc['gated SEIR-GNN (AAGCN) vs SEIR-LSTM, 9 origins', 'val delta']:+.2f}, test "
      f"{cmp_.loc['gated SEIR-GNN (AAGCN) vs SEIR-LSTM, 9 origins', 'test delta']:+.2f}")
print(f"  finished in {RESULTS['hours']:.2f} h; outputs in {OUT}")